# Training

In [2]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import webdataset as wds
import json
import time
import csv
import random
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. HARDWARE & PATH INITIALIZATION
# ==========================================
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    Usage: python train_model.py --gpu 1   OR   GPU_ID=1 python train_model.py
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

GPU_ID = select_gpu()
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")

if torch.cuda.is_available():
    print(f"[SYSTEM] PyTorch initialized. Active Device: {torch.cuda.get_device_name(GPU_ID)} (cuda:{GPU_ID})")
    print(f"[SYSTEM] CUDA Version: {torch.version.cuda} | PyTorch Version: {torch.__version__}")
    print(f"[SYSTEM] GPU Memory: {torch.cuda.get_device_properties(GPU_ID).total_memory / 1024**3:.1f} GB")
else:
    print(f"[WARNING] PyTorch could not find a CUDA device. Falling back to CPU.")

# --- Reproducibility ---
MASTER_SEED = 42
torch.manual_seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
random.seed(MASTER_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(MASTER_SEED)
    torch.cuda.manual_seed_all(MASTER_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"[SYSTEM] All RNG sources seeded with MASTER_SEED={MASTER_SEED}")
print(f"[SYSTEM] cudnn.deterministic={torch.backends.cudnn.deterministic}, cudnn.benchmark={torch.backends.cudnn.benchmark}")

user_prefix = input("\nEnter version prefix (e.g., tcn_v1) [Press Enter for default 'unknown_version']: ").strip()
VERSION_PREFIX = user_prefix if user_prefix else "unknown_version"

PROJECT_DIR = Path(os.path.expanduser("~/Capstone/Implementation"))
LOCAL_SHARDS_DIR = PROJECT_DIR / "data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards"
SAVE_DIR = PROJECT_DIR / f"models/{VERSION_PREFIX}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SCALARS_PATH = LOCAL_SHARDS_DIR / "normalization_scalars.json"

prefix = "/home/sanke24839/"

print(f"\n[PATHS] PROJECT_DIR     : {str(PROJECT_DIR).replace(prefix, '')}")
print(f"[PATHS] LOCAL_SHARDS_DIR: {str(LOCAL_SHARDS_DIR).replace(prefix, '')}")
print(f"[PATHS] SAVE_DIR        : {str(SAVE_DIR).replace(prefix, '')}")
print(f"[PATHS] SCALARS_PATH    : {str(SCALARS_PATH).replace(prefix, '')}")

with open(SCALARS_PATH, 'r') as f:
    scalars = json.load(f)

print(f"\n[DATA] Normalization scalars loaded successfully.")
print(f"[DATA] Feature Order: {scalars['feature_order']}")
print(f"[DATA] Number of Features: {len(scalars['feature_order'])}")

for i, feat in enumerate(scalars['feature_order']):
    print(f"[DATA]   [{i}] {feat:>30s} | min={scalars['global_minimums'][feat]:>12.4f} | max={scalars['global_maximums'][feat]:>12.4f}")

mins = torch.tensor([scalars['global_minimums'][c] for c in scalars['feature_order']], device=device)
maxs = torch.tensor([scalars['global_maximums'][c] for c in scalars['feature_order']], device=device)
ranges = maxs - mins
ranges[ranges == 0] = 1.0

print(f"\n[DATA] mins   tensor: shape={mins.shape}, dtype={mins.dtype}, device={mins.device}")
print(f"[DATA]   values: {mins.cpu().numpy()}")
print(f"[DATA] maxs   tensor: shape={maxs.shape}, dtype={maxs.dtype}, device={maxs.device}")
print(f"[DATA]   values: {maxs.cpu().numpy()}")
print(f"[DATA] ranges tensor: shape={ranges.shape}")
print(f"[DATA]   values: {ranges.cpu().numpy()}")

# ==========================================
# 2. ARCHITECTURE & DATALOADERS
# ==========================================
DEFAULT_EMPIRICAL = {
    "C0": 143.22, "C1": 140.25, "h0": 4.8713, "h1": 5.3871,
    "k01": 0.078038, "k10": 0.028120, "q0": 35.0, "q1": 35.0,
    "T_amb": 30.5
}

DEFAULT_FEATURE_INDICES = {
    "P0": 0, "P1": 1, "T0": 4, "T1": 5
}

BATCH_SIZE = 16384
NUM_WORKERS = 8
TOTAL_TRAIN_BATCHES = 10411
TOTAL_VAL_BATCHES = 1201

# Corrected DT from actual timestamp intervals in the dataset
# Measured intervals: 0.109, 0.107, 0.114, 0.108, 0.112, 0.108, 0.113, 0.108, 0.112
# Mean ≈ 0.11s
DT = 0.11

print(f"\n[CONFIG] BATCH_SIZE={BATCH_SIZE}, NUM_WORKERS={NUM_WORKERS}")
print(f"[CONFIG] TOTAL_TRAIN_BATCHES={TOTAL_TRAIN_BATCHES}, TOTAL_VAL_BATCHES={TOTAL_VAL_BATCHES}")
print(f"[CONFIG] DT={DT} (corrected from actual timestamp intervals)")
print(f"[CONFIG] Feature Indices: {DEFAULT_FEATURE_INDICES}")

print(f"\n[PHYSICS] Empirical parameter values (from prior calculation):")
for k, v in DEFAULT_EMPIRICAL.items():
    if k in ('q0', 'q1'):
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (unbounded via softplus, init guess)")
    elif k == 'T_amb':
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (bounds: [28.0, 33.0])")
    else:
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (bounds: [{v*0.90:.6f}, {v*1.10:.6f}])")


class DilatedTCN(nn.Module):
    def __init__(self, num_inputs=6, num_channels=[32, 64, 128], kernel_size=3):
        super(DilatedTCN, self).__init__()
        layers = []
        in_channels = num_inputs
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            padding = (kernel_size - 1) * dilation_size
            layers += [
                nn.ConstantPad1d((padding, 0), 0.0),
                nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
            ]
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1])


class PINNEngine(nn.Module):
    def __init__(self, empirical_params=None, feature_indices=None):
        super(PINNEngine, self).__init__()
        self.tcn = DilatedTCN()
        self.empirical = empirical_params if empirical_params is not None else dict(DEFAULT_EMPIRICAL)
        self.feature_indices = feature_indices if feature_indices is not None else dict(DEFAULT_FEATURE_INDICES)
        
        self.IDX_P0 = self.feature_indices["P0"]
        self.IDX_P1 = self.feature_indices["P1"]
        self.IDX_T0 = self.feature_indices["T0"]
        self.IDX_T1 = self.feature_indices["T1"]
        
        self.raw_params = nn.ParameterDict()
        for k in self.empirical:
            if k in ('q0', 'q1'):
                init_val = float(np.log(np.exp(self.empirical[k]) - 1.0))
            else:
                init_val = 0.0
            self.raw_params[k] = nn.Parameter(torch.tensor(init_val))

    def get_bounded_physics(self):
        phys = {}
        for k, exact_val in self.empirical.items():
            if k in ('q0', 'q1'):
                phys[k] = nn.functional.softplus(self.raw_params[k])
            elif k == 'T_amb':
                # Bound ambient temperature between 28.0°C and 33.0°C
                phys[k] = 28.0 + (33.0 - 28.0) * torch.sigmoid(self.raw_params[k])
            else:
                min_bound = exact_val * 0.90
                max_bound = exact_val * 1.10
                phys[k] = min_bound + (max_bound - min_bound) * torch.sigmoid(self.raw_params[k])
        return phys

    def forward(self, x):
        return self.tcn(x)


# --- Model Interpretability ---
model_temp = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES)
print(f"\n[MODEL] ========== FULL ARCHITECTURE ==========")
print(model_temp)
total_params = sum(p.numel() for p in model_temp.parameters())
trainable_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
tcn_params = sum(p.numel() for p in model_temp.tcn.parameters())
phys_params = sum(p.numel() for p in model_temp.raw_params.values())
print(f"\n[MODEL] Total Parameters     : {total_params:,}")
print(f"[MODEL] Trainable Parameters : {trainable_params:,}")
print(f"[MODEL] TCN (neural) Params  : {tcn_params:,}")
print(f"[MODEL] Physics Params       : {phys_params} ({list(DEFAULT_EMPIRICAL.keys())})")

print(f"\n[MODEL] Layer-by-layer breakdown:")
for name, param in model_temp.named_parameters():
    print(f"[MODEL]   {name:>40s} | shape={str(list(param.shape)):>15s} | numel={param.numel():>6d} | requires_grad={param.requires_grad}")

print(f"\n[MODEL] Initial physics parameter values:")
init_phys = model_temp.get_bounded_physics()
for k in DEFAULT_EMPIRICAL:
    raw = model_temp.raw_params[k].item()
    bounded = init_phys[k].item()
    if k in ('q0', 'q1'):
        print(f"[MODEL]   {k:>3s}: raw={raw:.6f} → softplus → {bounded:.6f} (init guess={DEFAULT_EMPIRICAL[k]:.6f})")
    else:
        sig = torch.sigmoid(model_temp.raw_params[k]).item()
        print(f"[MODEL]   {k:>3s}: raw={raw:.6f} → sigmoid={sig:.4f} → bounded={bounded:.6f} (empirical={DEFAULT_EMPIRICAL[k]:.6f})")
del model_temp, init_phys


def make_windows_vectorized(src, window_size=50, stride=10):
    """Yields (key, window) for WebDataset batching compatibility."""
    for key, npy_array in src:
        n_rows = npy_array.shape[0]
        if n_rows < window_size:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(npy_array, window_shape=window_size, axis=0)
        windows = np.swapaxes(windows, 1, 2)
        windows = windows[::stride].copy()
        for w in windows:
            yield key, w


def create_dataloader(shard_list, is_train=True):
    shard_shuffle_val = 100 if is_train else 0
    dataset = wds.WebDataset(shard_list, shardshuffle=shard_shuffle_val).decode().to_tuple("__key__", "data.npy")
    dataset = dataset.compose(lambda src: make_windows_vectorized(src, window_size=50, stride=10))
    if is_train:
        dataset = dataset.shuffle(5000)
    dataset = dataset.batched(BATCH_SIZE)
    return torch.utils.data.DataLoader(dataset, batch_size=None, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)


train_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "train").glob("*.tar"))]
val_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "val").glob("*.tar"))]
print(f"\n[DATA] Train shards: {len(train_files)} files")
print(f"[DATA] Val shards  : {len(val_files)} files")
if train_files:
    print(f"[DATA]   First train shard: {Path(train_files[0]).name}")
    print(f"[DATA]   Last  train shard: {Path(train_files[-1]).name}")

train_loader = create_dataloader(train_files, is_train=True)
val_loader = create_dataloader(val_files, is_train=False)


# ==========================================
# 3. INTEGRATED-FORM PHYSICS LOSS (FP32)
# ==========================================
def compute_physics_targets_norm(model, X_seq_abs, DT, mins_t, ranges_t):
    """
    Computes physics-predicted next temperature in NORMALIZED [0,1] space.
    
    INTEGRATED FORM (avoids noisy dT/dt from integer temperatures):
        T_next_physics = T_current + DT * (1/C) * (P + k*P' - h*(T - T_amb) + q)
    
    Then normalize:
        T_next_physics_norm = (T_next_physics - min) / range
    
    Both data loss and physics loss are MSE in [0,1] space → inherently comparable.
    """
    phys = model.get_bounded_physics()
    X_last = X_seq_abs[:, -1, :].float()
    P0  = X_last[:, model.IDX_P0]
    P1  = X_last[:, model.IDX_P1]
    T0_t = X_last[:, model.IDX_T0]
    T1_t = X_last[:, model.IDX_T1]

    # Integrated form: T_next = T_current + DT * RHS
    T0_phys = T0_t + DT * (1 / phys["C0"]) * (
        P0 + phys["k01"] * P1 - phys["h0"] * (T0_t - phys["T_amb"]) + phys["q0"]
    )
    T1_phys = T1_t + DT * (1 / phys["C1"]) * (
        P1 + phys["k10"] * P0 - phys["h1"] * (T1_t - phys["T_amb"]) + phys["q1"]
    )

    # Normalize to [0,1] using same scalars as data targets
    T0_phys_norm = (T0_phys - mins_t[4]) / ranges_t[4]
    T1_phys_norm = (T1_phys - mins_t[5]) / ranges_t[5]

    T_phys_norm = torch.stack([T0_phys_norm, T1_phys_norm], dim=1)  # [B, 2]

    return T_phys_norm, phys


# ==========================================
# 4. ReLoBRaLo ADAPTIVE LAMBDA
# ==========================================
class ReLoBRaLo:
    """
    Relative Loss Balancing with Random Lookback (ReLoBRaLo)
    Based on: Bischof & Kraus, 2021.
    """
    def __init__(self, num_losses=2, alpha=0.999, temperature=1.0, rho=0.999):
        self.num_losses = num_losses
        self.alpha = alpha
        self.temperature = temperature
        self.rho = rho
        self.initial_losses = None
        self.previous_losses = None
        self.weights = [1.0] * num_losses

    def get_weights(self, current_losses):
        assert len(current_losses) == self.num_losses
        if self.initial_losses is None:
            self.initial_losses = list(current_losses)
            self.previous_losses = list(current_losses)
            return list(self.weights)

        use_initial = random.random() < self.rho
        reference = self.initial_losses if use_initial else self.previous_losses

        ratios = []
        for i in range(self.num_losses):
            ref_val = reference[i] if reference[i] > 1e-12 else 1e-12
            ratios.append(current_losses[i] / ref_val)

        max_ratio = max(ratios)
        exp_ratios = [np.exp((r - max_ratio) / self.temperature) for r in ratios]
        sum_exp = sum(exp_ratios)
        instant_weights = [(self.num_losses * e / sum_exp) for e in exp_ratios]

        self.weights = [
            self.alpha * w_old + (1.0 - self.alpha) * w_new
            for w_old, w_new in zip(self.weights, instant_weights)
        ]
        self.previous_losses = list(current_losses)
        return list(self.weights)

    def state_dict(self):
        return {
            'initial_losses': self.initial_losses, 'previous_losses': self.previous_losses,
            'weights': self.weights, 'alpha': self.alpha,
            'temperature': self.temperature, 'rho': self.rho,
        }

    def load_state_dict(self, state):
        self.initial_losses = state['initial_losses']
        self.previous_losses = state['previous_losses']
        self.weights = state['weights']
        self.alpha = state['alpha']
        self.temperature = state['temperature']
        self.rho = state['rho']


# ==========================================
# 5. EPOCH 1 WALKTHROUGH (extracted)
# ==========================================
def epoch1_walkthrough(model, relobralo, current_phys, DT,
                       num_train_batches, total_train_loss, total_train_loss_data, total_train_loss_phys,
                       avg_train_loss, avg_train_data, avg_train_phys,
                       num_val_batches, total_val_loss, total_val_loss_data, total_val_loss_phys,
                       avg_val_loss, avg_val_data, avg_val_phys,
                       lambda_data, lambda_phys,
                       Y_val_pred_norm, Y_val_true_norm, T_phys_norm_v,
                       X_val, loss_data_v, loss_phys_v,
                       mins_cpu, ranges_cpu):
    """Complete mathematical walkthrough printed after Epoch 1 for verification."""
    SEP = "=" * 90
    print(f"\n[MATH] {SEP}")
    print(f"[MATH] >>> EPOCH 1: COMPLETE MATHEMATICAL WALKTHROUGH & PROOF <<<")
    print(f"[MATH] {SEP}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 1: TRAINING LOSS AGGREGATION ---")
    print(f"[MATH] Total train batches processed: {num_train_batches}")
    print(f"[MATH]   total_train_loss      = {total_train_loss:.8f}")
    print(f"[MATH]   total_train_loss_data  = {total_train_loss_data:.8f}")
    print(f"[MATH]   total_train_loss_phys  = {total_train_loss_phys:.8f}")
    print(f"[MATH] Averaging:")
    print(f"[MATH]   avg_train_loss = {total_train_loss:.8f} / {num_train_batches} = {avg_train_loss:.8f}")
    print(f"[MATH]   avg_train_data = {total_train_loss_data:.8f} / {num_train_batches} = {avg_train_data:.8f}")
    print(f"[MATH]   avg_train_phys = {total_train_loss_phys:.8f} / {num_train_batches} = {avg_train_phys:.8f}")
    print(f"[MATH]   NOTE: Physics loss uses INTEGRATED FORM in normalized [0,1] space.")
    print(f"[MATH]         Both losses are MSE in [0,1] → inherently comparable.")
    print(f"[MATH]   Final lambdas: λ_data={lambda_data:.8f}, λ_phys={lambda_phys:.8e}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 2: VALIDATION LOSS ---")
    print(f"[MATH] Val batches: {num_val_batches}, λ_data={lambda_data:.8f}, λ_phys={lambda_phys:.8e}")
    print(f"[MATH]   avg_val_loss = {avg_val_loss:.8f}")
    print(f"[MATH]   avg_val_data = {avg_val_data:.8f}")
    print(f"[MATH]   avg_val_phys = {avg_val_phys:.8f}")
    reconstructed_val = lambda_data * avg_val_data + lambda_phys * avg_val_phys
    match_str = "✓ MATCH" if abs(reconstructed_val - avg_val_loss) < 1e-6 else "✗ MISMATCH"
    print(f"[MATH]   Reconstruction: λd*data + λp*phys = {reconstructed_val:.8f} vs {avg_val_loss:.8f} → {match_str}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 3: DATA LOSS (last val batch, Row 0) ---")
    print(f"[MATH]   Y_pred_norm[0] = [{Y_val_pred_norm[0, 0].item():.8f}, {Y_val_pred_norm[0, 1].item():.8f}]")
    print(f"[MATH]   Y_true_norm[0] = [{Y_val_true_norm[0, 0].item():.8f}, {Y_val_true_norm[0, 1].item():.8f}]")
    d0 = Y_val_pred_norm[0, 0].item() - Y_val_true_norm[0, 0].item()
    d1 = Y_val_pred_norm[0, 1].item() - Y_val_true_norm[0, 1].item()
    print(f"[MATH]   Diffs: T0={d0:.8f}, T1={d1:.8f}")
    print(f"[MATH]   Squared: T0²={d0**2:.8f}, T1²={d1**2:.8f}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 4: INTEGRATED-FORM PHYSICS LOSS (Row 0) ---")
    print(f"[MATH]   FORMULA: T_phys = T_current + DT * (1/C) * (P + k*P' - h*(T-Tamb) + q)")
    print(f"[MATH]            T_phys_norm = (T_phys - min) / range")
    print(f"[MATH]            loss_phys = MSE(Y_pred_norm, T_phys_norm)")
    X_val_last = X_val[:, -1, :]
    p0 = X_val_last[0, model.IDX_P0].item()
    p1 = X_val_last[0, model.IDX_P1].item()
    t0_t = X_val_last[0, model.IDX_T0].item()
    t1_t = X_val_last[0, model.IDX_T1].item()

    C0, C1 = current_phys['C0'].item(), current_phys['C1'].item()
    h0, h1 = current_phys['h0'].item(), current_phys['h1'].item()
    k01, k10 = current_phys['k01'].item(), current_phys['k10'].item()
    q0, q1 = current_phys['q0'].item(), current_phys['q1'].item()
    t_amb = current_phys['T_amb'].item()

    print(f"[MATH]   Inputs: P0={p0:.4f}W, P1={p1:.4f}W, T0_t={t0_t:.4f}°C, T1_t={t1_t:.4f}°C, T_amb={t_amb:.4f}°C")

    print(f"[MATH]   Physics params:")
    for k in model.empirical:
        pct = ((current_phys[k].item() / model.empirical[k]) - 1) * 100
        ptype = "softplus" if k in ('q0', 'q1') else "sigmoid"
        print(f"[MATH]     {k:>3s} = {current_phys[k].item():.6f} (emp={model.empirical[k]:.6f}, Δ={pct:+.2f}%, {ptype})")

    # GPU0 manual calculation
    rhs0 = (1/C0) * (p0 + k01*p1 - h0*(t0_t - t_amb) + q0)
    t0_phys = t0_t + DT * rhs0
    t0_phys_norm = (t0_phys - mins_cpu[4].item()) / ranges_cpu[4].item()
    print(f"[MATH]   GPU0: RHS = (1/{C0:.4f}) * ({p0:.4f} + {k01:.6f}*{p1:.4f} - {h0:.4f}*({t0_t:.4f}-{t_amb:.4f}) + {q0:.4f})")
    print(f"[MATH]         RHS = {rhs0:.6f} °C/s")
    print(f"[MATH]         T0_phys = {t0_t:.4f} + {DT} * {rhs0:.6f} = {t0_phys:.6f} °C")
    print(f"[MATH]         T0_phys_norm = ({t0_phys:.6f} - {mins_cpu[4].item():.4f}) / {ranges_cpu[4].item():.4f} = {t0_phys_norm:.8f}")
    print(f"[MATH]         Tensor T_phys_norm[0,0] = {T_phys_norm_v[0, 0].item():.8f}")
    print(f"[MATH]         Y_pred_norm[0,0]        = {Y_val_pred_norm[0, 0].item():.8f}")
    dp0 = Y_val_pred_norm[0, 0].item() - T_phys_norm_v[0, 0].item()
    print(f"[MATH]         Physics diff: {dp0:.8f}, squared: {dp0**2:.8f}")

    # GPU1 manual calculation
    rhs1 = (1/C1) * (p1 + k10*p0 - h1*(t1_t - t_amb) + q1)
    t1_phys = t1_t + DT * rhs1
    t1_phys_norm = (t1_phys - mins_cpu[5].item()) / ranges_cpu[5].item()
    print(f"[MATH]   GPU1: RHS = (1/{C1:.4f}) * ({p1:.4f} + {k10:.6f}*{p0:.4f} - {h1:.4f}*({t1_t:.4f}-{t_amb:.4f}) + {q1:.4f})")
    print(f"[MATH]         RHS = {rhs1:.6f} °C/s")
    print(f"[MATH]         T1_phys = {t1_t:.4f} + {DT} * {rhs1:.6f} = {t1_phys:.6f} °C")
    print(f"[MATH]         T1_phys_norm = ({t1_phys:.6f} - {mins_cpu[5].item():.4f}) / {ranges_cpu[5].item():.4f} = {t1_phys_norm:.8f}")
    print(f"[MATH]         Tensor T_phys_norm[0,1] = {T_phys_norm_v[0, 1].item():.8f}")
    print(f"[MATH]         Y_pred_norm[0,1]        = {Y_val_pred_norm[0, 1].item():.8f}")
    dp1 = Y_val_pred_norm[0, 1].item() - T_phys_norm_v[0, 1].item()
    print(f"[MATH]         Physics diff: {dp1:.8f}, squared: {dp1**2:.8f}")

    print(f"[MATH]")
    print(f"[MATH]   Last batch loss_data = {loss_data_v.item():.8f}")
    print(f"[MATH]   Last batch loss_phys = {loss_phys_v.item():.8f}")
    print(f"[MATH]   RATIO: loss_phys / loss_data = {loss_phys_v.item() / max(loss_data_v.item(), 1e-12):.2f}x")
    print(f"[MATH]   (Should be within ~0.1x to ~10x for effective balancing)")

    print(f"[MATH]")
    print(f"[MATH] --- PART 5: ReLoBRaLo ---")
    print(f"[MATH]   Initial: data={relobralo.initial_losses[0]:.8f}, phys={relobralo.initial_losses[1]:.8f}")
    print(f"[MATH]   Final:   λ_data={relobralo.weights[0]:.8f}, λ_phys={relobralo.weights[1]:.8f}")
    print(f"[MATH]   Sum = {relobralo.weights[0] + relobralo.weights[1]:.6f} (should ≈ 2.0)")

    print(f"[MATH] {SEP}\n")


# ==========================================
# 6. TRAINING INFRASTRUCTURE & AUTO-RESUME
# ==========================================
EPOCHS = 50
LEARNING_RATE = 1e-3

print(f"\n[CONFIG] EPOCHS={EPOCHS}, LEARNING_RATE={LEARNING_RATE}, DT={DT}")
print(f"[CONFIG] Physics loss: INTEGRATED FORM in normalized [0,1] space")

model = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6)
mse_loss = nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

relobralo = ReLoBRaLo(num_losses=2, alpha=0.999, temperature=1.0, rho=0.999)
print(f"[CONFIG] ReLoBRaLo: alpha={relobralo.alpha}, temperature={relobralo.temperature}, rho={relobralo.rho}")
print(f"[CONFIG] Optimizer: Adam (lr={LEARNING_RATE})")
print(f"[CONFIG] Scheduler: ReduceLROnPlateau (factor=0.5, patience=4, min_lr=1e-6)")
print(f"[CONFIG] GradScaler: enabled (mixed precision FP16/FP32)")
print(f"[CONFIG] Gradient Clipping: max_norm=1.0")

start_epoch = 1
best_val_loss = float('inf')
RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
gpu_log_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu_stats.csv"
epoch_log_path = SAVE_DIR / f"{VERSION_PREFIX}_epoch_logs.csv"

if RESUME_PATH.exists():
    print(f"\n[SYSTEM] Found existing checkpoint at: {str(RESUME_PATH).replace(prefix, '')}")
    print(f"[SYSTEM] Resuming training...")
    checkpoint = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    if 'relobralo_state' in checkpoint:
        relobralo.load_state_dict(checkpoint['relobralo_state'])
        print(f"[SYSTEM] ReLoBRaLo restored: λ_data={relobralo.weights[0]:.6f}, λ_phys={relobralo.weights[1]:.6f}")
    torch.set_rng_state(checkpoint['torch_rng'].cpu().to(torch.uint8) if isinstance(checkpoint['torch_rng'], torch.Tensor) else torch.ByteTensor(checkpoint['torch_rng']))
    np.random.set_state(checkpoint['numpy_rng'])
    if torch.cuda.is_available() and 'cuda_rng' in checkpoint and checkpoint['cuda_rng'] is not None:
        cuda_rng = checkpoint['cuda_rng']
        if isinstance(cuda_rng, torch.Tensor):
            cuda_rng = cuda_rng.cpu().to(torch.uint8)
        else:
            cuda_rng = torch.ByteTensor(cuda_rng)
        torch.cuda.set_rng_state(cuda_rng)
    print(f"[SYSTEM] Resumed from Epoch {checkpoint['epoch']}. Best Val Loss: {best_val_loss:.6f}")
else:
    print(f"\n[SYSTEM] No checkpoint found. Starting fresh training.")

gpu_log_file = open(gpu_log_path, "w")
gpu_logger = subprocess.Popen(
    ["nvidia-smi", "-i", str(GPU_ID), "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
     "--format=csv", "-l", "5"],
    stdout=gpu_log_file, stderr=subprocess.STDOUT
)

csv_mode = 'a' if RESUME_PATH.exists() and epoch_log_path.exists() else 'w'
epoch_log_file = open(epoch_log_path, mode=csv_mode, newline='')
epoch_writer = csv.writer(epoch_log_file)
if csv_mode == 'w':
    epoch_writer.writerow([
        'Epoch', 'Train_Loss_Total', 'Train_Loss_Data', 'Train_Loss_Phys',
        'Val_Loss_Total', 'Val_Loss_Data', 'Val_Loss_Phys',
        'Lambda_Data', 'Lambda_Phys',
        'C0', 'C1', 'h0', 'h1', 'k01', 'k10', 'q0', 'q1', 'T_amb'
    ])

# ==========================================
# 7. PRE-TRAINING DATA INSPECTION
# ==========================================
print(f"\n[INSPECT] ========== FIRST BATCH DEEP INSPECTION ==========")
_inspect_keys, _inspect_block = next(iter(train_loader))
print(f"[INSPECT] raw data_block  : shape={list(_inspect_block.shape)}, dtype={_inspect_block.dtype}")
print(f"[INSPECT]   Interpretation: ({_inspect_block.shape[0]} samples) x ({_inspect_block.shape[1]} timesteps) x ({_inspect_block.shape[2]} features)")
print(f"[INSPECT]   Features: {scalars['feature_order']}")
print(f"[INSPECT] First sample, first 3 timesteps (RAW ABSOLUTE):")
for _t in range(min(3, _inspect_block.shape[1])):
    _row = _inspect_block[0, _t, :].numpy()
    print(f"[INSPECT]   t={_t}:")
    print(f"[INSPECT]       Power : GPU0={_row[0]:>7.4f}W | GPU1={_row[1]:>7.4f}W")
    print(f"[INSPECT]       Util  : GPU0={_row[2]:>7.4f}% | GPU1={_row[3]:>7.4f}%")
    print(f"[INSPECT]       Temp  : GPU0={_row[4]:>7.4f}C | GPU1={_row[5]:>7.4f}C")

_X_seq = _inspect_block[:, :-1, :]
_Y_true_abs = _inspect_block[:, -1, 4:6]
_Y_true_norm = (_Y_true_abs - mins[4:6].cpu()) / ranges[4:6].cpu()
_X_norm = (_X_seq - mins.cpu()) / ranges.cpu()

print(f"\n[INSPECT] After slicing & normalization:")
print(f"[INSPECT]   X_seq (input)      : shape={list(_X_seq.shape)}, dtype={_X_seq.dtype}")
print(f"[INSPECT]   Y_true_abs (target): shape={list(_Y_true_abs.shape)}, dtype={_Y_true_abs.dtype}")
print(f"[INSPECT]   X_norm (normalized): shape={list(_X_norm.shape)}, dtype={_X_norm.dtype}")
print(f"[INSPECT]   Y_true_norm        : shape={list(_Y_true_norm.shape)}, dtype={_Y_true_norm.dtype}")
print(f"[INSPECT] Sample Row 0:")
print(f"[INSPECT]   Absolute   : T0={_Y_true_abs[0, 0].item():.4f}, T1={_Y_true_abs[0, 1].item():.4f}")
print(f"[INSPECT]   Normalized : T0={_Y_true_norm[0, 0].item():.6f}, T1={_Y_true_norm[0, 1].item():.6f}")
print(f"[INSPECT] ========== END FIRST BATCH INSPECTION ==========\n")

del _inspect_keys, _inspect_block, _X_seq, _Y_true_abs, _Y_true_norm, _X_norm, _row

print(f"\n{'='*80}")
print(f"[TRAINING] INITIATING HPC MODEL TRAINING [{VERSION_PREFIX}]")
print(f"[TRAINING] Using ReLoBRaLo adaptive loss balancing")
print(f"[TRAINING] Physics loss: INTEGRATED FORM in normalized [0,1] space")
print(f"[TRAINING] Epochs: {start_epoch} → {EPOCHS} | Device: {device}")
print(f"{'='*80}")

# ==========================================
# 8. MAIN EPOCH LOOP
# ==========================================
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start_time = time.time()
        model.train()

        total_train_loss = 0.0
        total_train_loss_data = 0.0
        total_train_loss_phys = 0.0
        num_train_batches = 0

        train_pbar = tqdm(train_loader, total=TOTAL_TRAIN_BATCHES, desc=f"Epoch {epoch:02d}/{EPOCHS} [Train]", leave=False)

        for batch_idx, (keys, data_block) in enumerate(train_pbar):
            X_seq = data_block[:, :-1, :].to(device)
            Y_true_abs = data_block[:, -1, 4:6].to(device)
            Y_true_norm = (Y_true_abs - mins[4:6]) / ranges[4:6]
            X_norm = (X_seq - mins) / ranges

            optimizer.zero_grad()

            # --- Forward pass (FP16 for data loss) ---
            with torch.amp.autocast('cuda'):
                Y_pred_norm = model(X_norm)
                loss_data = mse_loss(Y_pred_norm, Y_true_norm)

            # --- Physics loss: integrated form in normalized space (FP32) ---
            T_phys_norm, _ = compute_physics_targets_norm(model, X_seq, DT, mins, ranges)
            loss_phys = mse_loss(Y_pred_norm.float(), T_phys_norm)

            # --- ReLoBRaLo: get adaptive weights ---
            lambdas = relobralo.get_weights([loss_data.item(), loss_phys.item()])
            lambda_data = lambdas[0]
            lambda_phys = lambdas[1]

            # --- Composite loss ---
            loss_total = lambda_data * loss_data + lambda_phys * loss_phys

            scaler.scale(loss_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss_total.item()
            total_train_loss_data += loss_data.item()
            total_train_loss_phys += loss_phys.item()
            num_train_batches += 1

            train_pbar.set_postfix({
                "Loss": f"{loss_total.item():.4f}",
                "Ld": f"{loss_data.item():.2e}",
                "Lp": f"{loss_phys.item():.2e}",
                "λd": f"{lambda_data:.3f}",
                "λp": f"{lambda_phys:.3f}"
            })

        avg_train_loss = total_train_loss / num_train_batches
        avg_train_data = total_train_loss_data / num_train_batches
        avg_train_phys = total_train_loss_phys / num_train_batches

        # --- VALIDATION LOOP ---
        model.eval()
        total_val_loss = 0.0
        total_val_loss_data = 0.0
        total_val_loss_phys = 0.0
        num_val_batches = 0

        val_pbar = tqdm(val_loader, total=TOTAL_VAL_BATCHES, desc="Validation", leave=False)

        with torch.no_grad():
            for batch_idx, (keys, val_block) in enumerate(val_pbar):
                X_val = val_block[:, :-1, :].to(device)
                Y_val_true_abs = val_block[:, -1, 4:6].to(device)
                Y_val_true_norm = (Y_val_true_abs - mins[4:6]) / ranges[4:6]
                X_val_norm = (X_val - mins) / ranges

                with torch.amp.autocast('cuda'):
                    Y_val_pred_norm = model(X_val_norm)
                    loss_data_v = mse_loss(Y_val_pred_norm, Y_val_true_norm)

                T_phys_norm_v, _ = compute_physics_targets_norm(model, X_val, DT, mins, ranges)
                loss_phys_v = mse_loss(Y_val_pred_norm.float(), T_phys_norm_v)

                loss_data_v_scalar = loss_data_v.item()
                loss_phys_v_scalar = loss_phys_v.item()
                loss_total_v = lambda_data * loss_data_v_scalar + lambda_phys * loss_phys_v_scalar

                total_val_loss += loss_total_v
                total_val_loss_data += loss_data_v_scalar
                total_val_loss_phys += loss_phys_v_scalar
                num_val_batches += 1

        avg_val_loss = total_val_loss / num_val_batches
        avg_val_data = total_val_loss_data / num_val_batches
        avg_val_phys = total_val_loss_phys / num_val_batches
        epoch_duration = time.time() - epoch_start_time

        current_phys = model.get_bounded_physics()
        epoch_writer.writerow([
            epoch, avg_train_loss, avg_train_data, avg_train_phys,
            avg_val_loss, avg_val_data, avg_val_phys,
            lambda_data, lambda_phys,
            *[current_phys[k].item() for k in ['C0','C1','h0','h1','k01','k10','q0','q1','T_amb']]
        ])
        epoch_log_file.flush()

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_flag = f" [LR Reduced to {new_lr:.2e}]" if new_lr < old_lr else ""
        best_flag = ""

        checkpoint_state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
            'relobralo_state': relobralo.state_dict(),
            'torch_rng': torch.get_rng_state(),
            'numpy_rng': np.random.get_state(),
            'cuda_rng': torch.cuda.get_rng_state() if torch.cuda.is_available() else None
        }
        torch.save(checkpoint_state, RESUME_PATH)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            checkpoint_state['best_val_loss'] = best_val_loss
            torch.save(checkpoint_state, BEST_PATH)
            best_flag = " [*]"

        if epoch == 1:
            epoch1_walkthrough(
                model=model, relobralo=relobralo, current_phys=current_phys, DT=DT,
                num_train_batches=num_train_batches,
                total_train_loss=total_train_loss, total_train_loss_data=total_train_loss_data,
                total_train_loss_phys=total_train_loss_phys,
                avg_train_loss=avg_train_loss, avg_train_data=avg_train_data, avg_train_phys=avg_train_phys,
                num_val_batches=num_val_batches,
                total_val_loss=total_val_loss, total_val_loss_data=total_val_loss_data,
                total_val_loss_phys=total_val_loss_phys,
                avg_val_loss=avg_val_loss, avg_val_data=avg_val_data, avg_val_phys=avg_val_phys,
                lambda_data=lambda_data, lambda_phys=lambda_phys,
                Y_val_pred_norm=Y_val_pred_norm, Y_val_true_norm=Y_val_true_norm,
                T_phys_norm_v=T_phys_norm_v,
                X_val=X_val, loss_data_v=loss_data_v, loss_phys_v=loss_phys_v,
                mins_cpu=mins.cpu(), ranges_cpu=ranges.cpu()
            )

        print(f"[TRAINING] Epoch {epoch:02d}/{EPOCHS} | Time: {epoch_duration:.1f}s | "
              f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | "
              f"λd:{lambda_data:.3f} λp:{lambda_phys:.3f}{best_flag}{lr_flag}")

except KeyboardInterrupt:
    print("\n[!] Training manually interrupted. Checkpoints are safe.")
finally:
    try:
        gpu_logger.kill()
    except ProcessLookupError:
        pass
    gpu_log_file.close()
    epoch_log_file.close()
    print(f"[SYSTEM] Background GPU logging terminated. Epoch logs securely saved.")



[SYSTEM] Live GPU Status:
Thu Mar 26 18:34:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 30%   41C    P8              8W /  200W |    7395MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.
[SYSTEM] PyTorch initialized. Active Device: NVIDIA RTX A4500 (cuda:1)
[SYSTEM] CUDA Version: 12.6 | PyTorch Version: 2.9.1+cu126
[SYSTEM] GPU Memory: 19.6 GB
[SYSTEM] All RNG sources seeded with MASTER_SEED=42
[SYSTEM] cudnn.deterministic=True, cudnn.benchmark=False



Enter version prefix (e.g., tcn_v1) [Press Enter for default 'unknown_version']:  TCN_ReLoBRaLo_v2



[PATHS] PROJECT_DIR     : Capstone/Implementation
[PATHS] LOCAL_SHARDS_DIR: Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards
[PATHS] SAVE_DIR        : Capstone/Implementation/models/TCN_ReLoBRaLo_v2
[PATHS] SCALARS_PATH    : Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards/normalization_scalars.json

[DATA] Normalization scalars loaded successfully.
[DATA] Feature Order: ['power_draw_gpu_0_W', 'power_draw_gpu_1_W', 'utilization_gpu_0_pct', 'utilization_gpu_1_pct', 'temperature_gpu_0', 'temperature_gpu_1']
[DATA] Number of Features: 6
[DATA]   [0]             power_draw_gpu_0_W | min=     21.8100 | max=    331.0800
[DATA]   [1]             power_draw_gpu_1_W | min=     22.7800 | max=    328.8600
[DATA]   [2]          utilization_gpu_0_pct | min=      0.0000 | max=    100.0000
[DATA]   [3]          utilization_gpu_1_pct | min=      0.0000 | max=    100.0000

...

[TRAINING] Epoch 02/50 | Time: 825.2s | Train Loss: 0.003435 | Val Loss: 0.003560 | λd:1.000 λp:1.000 [*]


[TRAINING] Epoch 03/50 | Time: 889.6s | Train Loss: 0.002539 | Val Loss: 0.004610 | λd:1.000 λp:1.000


[TRAINING] Epoch 04/50 | Time: 860.5s | Train Loss: 0.002029 | Val Loss: 0.003430 | λd:1.000 λp:1.000 [*]


[TRAINING] Epoch 05/50 | Time: 857.7s | Train Loss: 0.001817 | Val Loss: 0.003501 | λd:1.000 λp:1.000



[!] Training manually interrupted. Checkpoints are safe.
[SYSTEM] Background GPU logging terminated. Epoch logs securely saved.


In [1]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import webdataset as wds
import json
import time
import csv
import random
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. HARDWARE & PATH INITIALIZATION
# ==========================================
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    Usage: python train_model.py --gpu 1   OR   GPU_ID=1 python train_model.py
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

GPU_ID = select_gpu()
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")

if torch.cuda.is_available():
    print(f"[SYSTEM] PyTorch initialized. Active Device: {torch.cuda.get_device_name(GPU_ID)} (cuda:{GPU_ID})")
    print(f"[SYSTEM] CUDA Version: {torch.version.cuda} | PyTorch Version: {torch.__version__}")
    print(f"[SYSTEM] GPU Memory: {torch.cuda.get_device_properties(GPU_ID).total_memory / 1024**3:.1f} GB")
else:
    print(f"[WARNING] PyTorch could not find a CUDA device. Falling back to CPU.")

# --- Reproducibility ---
MASTER_SEED = 42
torch.manual_seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
random.seed(MASTER_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(MASTER_SEED)
    torch.cuda.manual_seed_all(MASTER_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"[SYSTEM] All RNG sources seeded with MASTER_SEED={MASTER_SEED}")
print(f"[SYSTEM] cudnn.deterministic={torch.backends.cudnn.deterministic}, cudnn.benchmark={torch.backends.cudnn.benchmark}")

user_prefix = input("\nEnter version prefix (e.g., tcn_v1) [Press Enter for default 'unknown_version']: ").strip()
VERSION_PREFIX = user_prefix if user_prefix else "unknown_version"

PROJECT_DIR = Path(os.path.expanduser("~/Capstone/Implementation"))
LOCAL_SHARDS_DIR = PROJECT_DIR / "data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards"
SAVE_DIR = PROJECT_DIR / f"models/{VERSION_PREFIX}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SCALARS_PATH = LOCAL_SHARDS_DIR / "normalization_scalars.json"

prefix = "/home/sanke24839/"

print(f"\n[PATHS] PROJECT_DIR     : {str(PROJECT_DIR).replace(prefix, '')}")
print(f"[PATHS] LOCAL_SHARDS_DIR: {str(LOCAL_SHARDS_DIR).replace(prefix, '')}")
print(f"[PATHS] SAVE_DIR        : {str(SAVE_DIR).replace(prefix, '')}")
print(f"[PATHS] SCALARS_PATH    : {str(SCALARS_PATH).replace(prefix, '')}")

with open(SCALARS_PATH, 'r') as f:
    scalars = json.load(f)

print(f"\n[DATA] Normalization scalars loaded successfully.")
print(f"[DATA] Feature Order: {scalars['feature_order']}")
print(f"[DATA] Number of Features: {len(scalars['feature_order'])}")

for i, feat in enumerate(scalars['feature_order']):
    print(f"[DATA]   [{i}] {feat:>30s} | min={scalars['global_minimums'][feat]:>12.4f} | max={scalars['global_maximums'][feat]:>12.4f}")

mins = torch.tensor([scalars['global_minimums'][c] for c in scalars['feature_order']], device=device)
maxs = torch.tensor([scalars['global_maximums'][c] for c in scalars['feature_order']], device=device)
ranges = maxs - mins
ranges[ranges == 0] = 1.0

print(f"\n[DATA] mins   tensor: shape={mins.shape}, dtype={mins.dtype}, device={mins.device}")
print(f"[DATA]   values: {mins.cpu().numpy()}")
print(f"[DATA] maxs   tensor: shape={maxs.shape}, dtype={maxs.dtype}, device={maxs.device}")
print(f"[DATA]   values: {maxs.cpu().numpy()}")
print(f"[DATA] ranges tensor: shape={ranges.shape}")
print(f"[DATA]   values: {ranges.cpu().numpy()}")

# ==========================================
# 2. ARCHITECTURE & DATALOADERS
# ==========================================
DEFAULT_EMPIRICAL = {
    "C0": 143.22, "C1": 140.25, "h0": 4.8713, "h1": 5.3871,
    "k01": 0.078038, "k10": 0.028120, "q0": 35.0, "q1": 35.0,
    "T_amb": 30.5
}

DEFAULT_FEATURE_INDICES = {
    "P0": 0, "P1": 1, "T0": 4, "T1": 5
}

BATCH_SIZE = 16384
NUM_WORKERS = 8
TOTAL_TRAIN_BATCHES = 10411
TOTAL_VAL_BATCHES = 1201

# Corrected DT from actual timestamp intervals in the dataset
# Measured intervals: 0.109, 0.107, 0.114, 0.108, 0.112, 0.108, 0.113, 0.108, 0.112
# Mean ≈ 0.11s
DT = 0.11

print(f"\n[CONFIG] BATCH_SIZE={BATCH_SIZE}, NUM_WORKERS={NUM_WORKERS}")
print(f"[CONFIG] TOTAL_TRAIN_BATCHES={TOTAL_TRAIN_BATCHES}, TOTAL_VAL_BATCHES={TOTAL_VAL_BATCHES}")
print(f"[CONFIG] DT={DT} (corrected from actual timestamp intervals)")
print(f"[CONFIG] Feature Indices: {DEFAULT_FEATURE_INDICES}")

print(f"\n[PHYSICS] Empirical parameter values (from prior calculation):")
for k, v in DEFAULT_EMPIRICAL.items():
    if k in ('q0', 'q1'):
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (unbounded via softplus, init guess)")
    elif k == 'T_amb':
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (bounds: [28.0, 33.0])")
    else:
        print(f"[PHYSICS]   {k:>3s} = {v:.6f}  (bounds: [{v*0.90:.6f}, {v*1.10:.6f}])")


class DilatedTCN(nn.Module):
    def __init__(self, num_inputs=6, num_channels=[32, 64, 128], kernel_size=3):
        super(DilatedTCN, self).__init__()
        layers = []
        in_channels = num_inputs
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            padding = (kernel_size - 1) * dilation_size
            layers += [
                nn.ConstantPad1d((padding, 0), 0.0),
                nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
            ]
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1])


class PINNEngine(nn.Module):
    def __init__(self, empirical_params=None, feature_indices=None):
        super(PINNEngine, self).__init__()
        self.tcn = DilatedTCN()
        self.empirical = empirical_params if empirical_params is not None else dict(DEFAULT_EMPIRICAL)
        self.feature_indices = feature_indices if feature_indices is not None else dict(DEFAULT_FEATURE_INDICES)
        
        self.IDX_P0 = self.feature_indices["P0"]
        self.IDX_P1 = self.feature_indices["P1"]
        self.IDX_T0 = self.feature_indices["T0"]
        self.IDX_T1 = self.feature_indices["T1"]
        
        self.raw_params = nn.ParameterDict()
        for k in self.empirical:
            if k in ('q0', 'q1'):
                init_val = float(np.log(np.exp(self.empirical[k]) - 1.0))
            else:
                init_val = 0.0
            self.raw_params[k] = nn.Parameter(torch.tensor(init_val))

    def get_bounded_physics(self):
        phys = {}
        for k, exact_val in self.empirical.items():
            if k in ('q0', 'q1'):
                phys[k] = nn.functional.softplus(self.raw_params[k])
            elif k == 'T_amb':
                # Bound ambient temperature between 28.0°C and 33.0°C
                phys[k] = 28.0 + (33.0 - 28.0) * torch.sigmoid(self.raw_params[k])
            else:
                min_bound = exact_val * 0.90
                max_bound = exact_val * 1.10
                phys[k] = min_bound + (max_bound - min_bound) * torch.sigmoid(self.raw_params[k])
        return phys

    def forward(self, x):
        return self.tcn(x)


# --- Model Interpretability ---
model_temp = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES)
print(f"\n[MODEL] ========== FULL ARCHITECTURE ==========")
print(model_temp)
total_params = sum(p.numel() for p in model_temp.parameters())
trainable_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
tcn_params = sum(p.numel() for p in model_temp.tcn.parameters())
phys_params = sum(p.numel() for p in model_temp.raw_params.values())
print(f"\n[MODEL] Total Parameters     : {total_params:,}")
print(f"[MODEL] Trainable Parameters : {trainable_params:,}")
print(f"[MODEL] TCN (neural) Params  : {tcn_params:,}")
print(f"[MODEL] Physics Params       : {phys_params} ({list(DEFAULT_EMPIRICAL.keys())})")

print(f"\n[MODEL] Layer-by-layer breakdown:")
for name, param in model_temp.named_parameters():
    print(f"[MODEL]   {name:>40s} | shape={str(list(param.shape)):>15s} | numel={param.numel():>6d} | requires_grad={param.requires_grad}")

print(f"\n[MODEL] Initial physics parameter values:")
init_phys = model_temp.get_bounded_physics()
for k in DEFAULT_EMPIRICAL:
    raw = model_temp.raw_params[k].item()
    bounded = init_phys[k].item()
    if k in ('q0', 'q1'):
        print(f"[MODEL]   {k:>3s}: raw={raw:.6f} → softplus → {bounded:.6f} (init guess={DEFAULT_EMPIRICAL[k]:.6f})")
    else:
        sig = torch.sigmoid(model_temp.raw_params[k]).item()
        print(f"[MODEL]   {k:>3s}: raw={raw:.6f} → sigmoid={sig:.4f} → bounded={bounded:.6f} (empirical={DEFAULT_EMPIRICAL[k]:.6f})")
del model_temp, init_phys


def make_windows_vectorized(src, window_size=50, stride=10):
    """Yields (key, window) for WebDataset batching compatibility."""
    for key, npy_array in src:
        n_rows = npy_array.shape[0]
        if n_rows < window_size:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(npy_array, window_shape=window_size, axis=0)
        windows = np.swapaxes(windows, 1, 2)
        windows = windows[::stride].copy()
        for w in windows:
            yield key, w


def create_dataloader(shard_list, is_train=True):
    shard_shuffle_val = 100 if is_train else 0
    dataset = wds.WebDataset(shard_list, shardshuffle=shard_shuffle_val).decode().to_tuple("__key__", "data.npy")
    dataset = dataset.compose(lambda src: make_windows_vectorized(src, window_size=50, stride=10))
    if is_train:
        dataset = dataset.shuffle(5000)
    dataset = dataset.batched(BATCH_SIZE)
    return torch.utils.data.DataLoader(dataset, batch_size=None, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)


train_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "train").glob("*.tar"))]
val_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "val").glob("*.tar"))]
print(f"\n[DATA] Train shards: {len(train_files)} files")
print(f"[DATA] Val shards  : {len(val_files)} files")
if train_files:
    print(f"[DATA]   First train shard: {Path(train_files[0]).name}")
    print(f"[DATA]   Last  train shard: {Path(train_files[-1]).name}")

train_loader = create_dataloader(train_files, is_train=True)
val_loader = create_dataloader(val_files, is_train=False)


# ==========================================
# 3. INTEGRATED-FORM PHYSICS LOSS (FP32)
# ==========================================
def compute_physics_targets_norm(model, X_seq_abs, DT, mins_t, ranges_t):
    """
    Computes physics-predicted next temperature in NORMALIZED [0,1] space.
    
    INTEGRATED FORM (avoids noisy dT/dt from integer temperatures):
        T_next_physics = T_current + DT * (1/C) * (P + k*P' - h*(T - T_amb) + q)
    
    Then normalize:
        T_next_physics_norm = (T_next_physics - min) / range
    
    Both data loss and physics loss are MSE in [0,1] space → inherently comparable.
    """
    phys = model.get_bounded_physics()
    X_last = X_seq_abs[:, -1, :].float()
    P0  = X_last[:, model.IDX_P0]
    P1  = X_last[:, model.IDX_P1]
    T0_t = X_last[:, model.IDX_T0]
    T1_t = X_last[:, model.IDX_T1]

    # Integrated form: T_next = T_current + DT * RHS
    T0_phys = T0_t + DT * (1 / phys["C0"]) * (
        P0 + phys["k01"] * P1 - phys["h0"] * (T0_t - phys["T_amb"]) + phys["q0"]
    )
    T1_phys = T1_t + DT * (1 / phys["C1"]) * (
        P1 + phys["k10"] * P0 - phys["h1"] * (T1_t - phys["T_amb"]) + phys["q1"]
    )

    # Normalize to [0,1] using same scalars as data targets
    T0_phys_norm = (T0_phys - mins_t[4]) / ranges_t[4]
    T1_phys_norm = (T1_phys - mins_t[5]) / ranges_t[5]

    T_phys_norm = torch.stack([T0_phys_norm, T1_phys_norm], dim=1)  # [B, 2]

    return T_phys_norm, phys


# ==========================================
# 4. ReLoBRaLo ADAPTIVE LAMBDA
# ==========================================
class ReLoBRaLo:
    """
    Relative Loss Balancing with Random Lookback (ReLoBRaLo)
    Based on: Bischof & Kraus, 2021.
    """
    def __init__(self, num_losses=2, alpha=0.999, temperature=1.0, rho=0.999):
        self.num_losses = num_losses
        self.alpha = alpha
        self.temperature = temperature
        self.rho = rho
        self.initial_losses = None
        self.previous_losses = None
        self.weights = [1.0] * num_losses

    def get_weights(self, current_losses):
        assert len(current_losses) == self.num_losses
        if self.initial_losses is None:
            self.initial_losses = list(current_losses)
            self.previous_losses = list(current_losses)
            return list(self.weights)

        use_initial = random.random() < self.rho
        reference = self.initial_losses if use_initial else self.previous_losses

        ratios = []
        for i in range(self.num_losses):
            ref_val = reference[i] if reference[i] > 1e-12 else 1e-12
            ratios.append(current_losses[i] / ref_val)

        max_ratio = max(ratios)
        exp_ratios = [np.exp((r - max_ratio) / self.temperature) for r in ratios]
        sum_exp = sum(exp_ratios)
        instant_weights = [(self.num_losses * e / sum_exp) for e in exp_ratios]

        self.weights = [
            self.alpha * w_old + (1.0 - self.alpha) * w_new
            for w_old, w_new in zip(self.weights, instant_weights)
        ]
        self.previous_losses = list(current_losses)
        return list(self.weights)

    def state_dict(self):
        return {
            'initial_losses': self.initial_losses, 'previous_losses': self.previous_losses,
            'weights': self.weights, 'alpha': self.alpha,
            'temperature': self.temperature, 'rho': self.rho,
        }

    def load_state_dict(self, state):
        self.initial_losses = state['initial_losses']
        self.previous_losses = state['previous_losses']
        self.weights = state['weights']
        self.alpha = state['alpha']
        self.temperature = state['temperature']
        self.rho = state['rho']


# ==========================================
# 5. EPOCH 1 WALKTHROUGH (extracted)
# ==========================================
def epoch1_walkthrough(model, relobralo, current_phys, DT,
                       num_train_batches, total_train_loss, total_train_loss_data, total_train_loss_phys,
                       avg_train_loss, avg_train_data, avg_train_phys,
                       num_val_batches, total_val_loss, total_val_loss_data, total_val_loss_phys,
                       avg_val_loss, avg_val_data, avg_val_phys,
                       lambda_data, lambda_phys,
                       Y_val_pred_norm, Y_val_true_norm, T_phys_norm_v,
                       X_val, loss_data_v, loss_phys_v,
                       mins_cpu, ranges_cpu):
    """Complete mathematical walkthrough printed after Epoch 1 for verification."""
    SEP = "=" * 90
    print(f"\n[MATH] {SEP}")
    print(f"[MATH] >>> EPOCH 1: COMPLETE MATHEMATICAL WALKTHROUGH & PROOF <<<")
    print(f"[MATH] {SEP}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 1: TRAINING LOSS AGGREGATION ---")
    print(f"[MATH] Total train batches processed: {num_train_batches}")
    print(f"[MATH]   total_train_loss      = {total_train_loss:.8f}")
    print(f"[MATH]   total_train_loss_data  = {total_train_loss_data:.8f}")
    print(f"[MATH]   total_train_loss_phys  = {total_train_loss_phys:.8f}")
    print(f"[MATH] Averaging:")
    print(f"[MATH]   avg_train_loss = {total_train_loss:.8f} / {num_train_batches} = {avg_train_loss:.8f}")
    print(f"[MATH]   avg_train_data = {total_train_loss_data:.8f} / {num_train_batches} = {avg_train_data:.8f}")
    print(f"[MATH]   avg_train_phys = {total_train_loss_phys:.8f} / {num_train_batches} = {avg_train_phys:.8f}")
    print(f"[MATH]   NOTE: Physics loss uses INTEGRATED FORM in normalized [0,1] space.")
    print(f"[MATH]         Both losses are MSE in [0,1] → inherently comparable.")
    print(f"[MATH]   Final lambdas: λ_data={lambda_data:.8f}, λ_phys={lambda_phys:.8e}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 2: VALIDATION LOSS ---")
    print(f"[MATH] Val batches: {num_val_batches}, λ_data={lambda_data:.8f}, λ_phys={lambda_phys:.8e}")
    print(f"[MATH]   avg_val_loss = {avg_val_loss:.8f}")
    print(f"[MATH]   avg_val_data = {avg_val_data:.8f}")
    print(f"[MATH]   avg_val_phys = {avg_val_phys:.8f}")
    reconstructed_val = lambda_data * avg_val_data + lambda_phys * avg_val_phys
    match_str = "✓ MATCH" if abs(reconstructed_val - avg_val_loss) < 1e-6 else "✗ MISMATCH"
    print(f"[MATH]   Reconstruction: λd*data + λp*phys = {reconstructed_val:.8f} vs {avg_val_loss:.8f} → {match_str}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 3: DATA LOSS (last val batch, Row 0) ---")
    print(f"[MATH]   Y_pred_norm[0] = [{Y_val_pred_norm[0, 0].item():.8f}, {Y_val_pred_norm[0, 1].item():.8f}]")
    print(f"[MATH]   Y_true_norm[0] = [{Y_val_true_norm[0, 0].item():.8f}, {Y_val_true_norm[0, 1].item():.8f}]")
    d0 = Y_val_pred_norm[0, 0].item() - Y_val_true_norm[0, 0].item()
    d1 = Y_val_pred_norm[0, 1].item() - Y_val_true_norm[0, 1].item()
    print(f"[MATH]   Diffs: T0={d0:.8f}, T1={d1:.8f}")
    print(f"[MATH]   Squared: T0²={d0**2:.8f}, T1²={d1**2:.8f}")

    print(f"[MATH]")
    print(f"[MATH] --- PART 4: INTEGRATED-FORM PHYSICS LOSS (Row 0) ---")
    print(f"[MATH]   FORMULA: T_phys = T_current + DT * (1/C) * (P + k*P' - h*(T-Tamb) + q)")
    print(f"[MATH]            T_phys_norm = (T_phys - min) / range")
    print(f"[MATH]            loss_phys = MSE(Y_pred_norm, T_phys_norm)")
    X_val_last = X_val[:, -1, :]
    p0 = X_val_last[0, model.IDX_P0].item()
    p1 = X_val_last[0, model.IDX_P1].item()
    t0_t = X_val_last[0, model.IDX_T0].item()
    t1_t = X_val_last[0, model.IDX_T1].item()

    C0, C1 = current_phys['C0'].item(), current_phys['C1'].item()
    h0, h1 = current_phys['h0'].item(), current_phys['h1'].item()
    k01, k10 = current_phys['k01'].item(), current_phys['k10'].item()
    q0, q1 = current_phys['q0'].item(), current_phys['q1'].item()
    t_amb = current_phys['T_amb'].item()

    print(f"[MATH]   Inputs: P0={p0:.4f}W, P1={p1:.4f}W, T0_t={t0_t:.4f}°C, T1_t={t1_t:.4f}°C, T_amb={t_amb:.4f}°C")

    print(f"[MATH]   Physics params:")
    for k in model.empirical:
        pct = ((current_phys[k].item() / model.empirical[k]) - 1) * 100
        ptype = "softplus" if k in ('q0', 'q1') else "sigmoid"
        print(f"[MATH]     {k:>3s} = {current_phys[k].item():.6f} (emp={model.empirical[k]:.6f}, Δ={pct:+.2f}%, {ptype})")

    # GPU0 manual calculation
    rhs0 = (1/C0) * (p0 + k01*p1 - h0*(t0_t - t_amb) + q0)
    t0_phys = t0_t + DT * rhs0
    t0_phys_norm = (t0_phys - mins_cpu[4].item()) / ranges_cpu[4].item()
    print(f"[MATH]   GPU0: RHS = (1/{C0:.4f}) * ({p0:.4f} + {k01:.6f}*{p1:.4f} - {h0:.4f}*({t0_t:.4f}-{t_amb:.4f}) + {q0:.4f})")
    print(f"[MATH]         RHS = {rhs0:.6f} °C/s")
    print(f"[MATH]         T0_phys = {t0_t:.4f} + {DT} * {rhs0:.6f} = {t0_phys:.6f} °C")
    print(f"[MATH]         T0_phys_norm = ({t0_phys:.6f} - {mins_cpu[4].item():.4f}) / {ranges_cpu[4].item():.4f} = {t0_phys_norm:.8f}")
    print(f"[MATH]         Tensor T_phys_norm[0,0] = {T_phys_norm_v[0, 0].item():.8f}")
    print(f"[MATH]         Y_pred_norm[0,0]        = {Y_val_pred_norm[0, 0].item():.8f}")
    dp0 = Y_val_pred_norm[0, 0].item() - T_phys_norm_v[0, 0].item()
    print(f"[MATH]         Physics diff: {dp0:.8f}, squared: {dp0**2:.8f}")

    # GPU1 manual calculation
    rhs1 = (1/C1) * (p1 + k10*p0 - h1*(t1_t - t_amb) + q1)
    t1_phys = t1_t + DT * rhs1
    t1_phys_norm = (t1_phys - mins_cpu[5].item()) / ranges_cpu[5].item()
    print(f"[MATH]   GPU1: RHS = (1/{C1:.4f}) * ({p1:.4f} + {k10:.6f}*{p0:.4f} - {h1:.4f}*({t1_t:.4f}-{t_amb:.4f}) + {q1:.4f})")
    print(f"[MATH]         RHS = {rhs1:.6f} °C/s")
    print(f"[MATH]         T1_phys = {t1_t:.4f} + {DT} * {rhs1:.6f} = {t1_phys:.6f} °C")
    print(f"[MATH]         T1_phys_norm = ({t1_phys:.6f} - {mins_cpu[5].item():.4f}) / {ranges_cpu[5].item():.4f} = {t1_phys_norm:.8f}")
    print(f"[MATH]         Tensor T_phys_norm[0,1] = {T_phys_norm_v[0, 1].item():.8f}")
    print(f"[MATH]         Y_pred_norm[0,1]        = {Y_val_pred_norm[0, 1].item():.8f}")
    dp1 = Y_val_pred_norm[0, 1].item() - T_phys_norm_v[0, 1].item()
    print(f"[MATH]         Physics diff: {dp1:.8f}, squared: {dp1**2:.8f}")

    print(f"[MATH]")
    print(f"[MATH]   Last batch loss_data = {loss_data_v.item():.8f}")
    print(f"[MATH]   Last batch loss_phys = {loss_phys_v.item():.8f}")
    print(f"[MATH]   RATIO: loss_phys / loss_data = {loss_phys_v.item() / max(loss_data_v.item(), 1e-12):.2f}x")
    print(f"[MATH]   (Should be within ~0.1x to ~10x for effective balancing)")

    print(f"[MATH]")
    print(f"[MATH] --- PART 5: ReLoBRaLo ---")
    print(f"[MATH]   Initial: data={relobralo.initial_losses[0]:.8f}, phys={relobralo.initial_losses[1]:.8f}")
    print(f"[MATH]   Final:   λ_data={relobralo.weights[0]:.8f}, λ_phys={relobralo.weights[1]:.8f}")
    print(f"[MATH]   Sum = {relobralo.weights[0] + relobralo.weights[1]:.6f} (should ≈ 2.0)")

    print(f"[MATH] {SEP}\n")


# ==========================================
# 6. TRAINING INFRASTRUCTURE & AUTO-RESUME
# ==========================================
EPOCHS = 50
LEARNING_RATE = 1e-3

print(f"\n[CONFIG] EPOCHS={EPOCHS}, LEARNING_RATE={LEARNING_RATE}, DT={DT}")
print(f"[CONFIG] Physics loss: INTEGRATED FORM in normalized [0,1] space")

model = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6)
mse_loss = nn.MSELoss()
scaler = torch.amp.GradScaler('cuda')

relobralo = ReLoBRaLo(num_losses=2, alpha=0.999, temperature=1.0, rho=0.999)
print(f"[CONFIG] ReLoBRaLo: alpha={relobralo.alpha}, temperature={relobralo.temperature}, rho={relobralo.rho}")
print(f"[CONFIG] Optimizer: Adam (lr={LEARNING_RATE})")
print(f"[CONFIG] Scheduler: ReduceLROnPlateau (factor=0.5, patience=4, min_lr=1e-6)")
print(f"[CONFIG] GradScaler: enabled (mixed precision FP16/FP32)")
print(f"[CONFIG] Gradient Clipping: max_norm=1.0")

start_epoch = 1
best_val_loss = float('inf')
RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
gpu_log_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu_stats.csv"
epoch_log_path = SAVE_DIR / f"{VERSION_PREFIX}_epoch_logs.csv"

if RESUME_PATH.exists():
    print(f"\n[SYSTEM] Found existing checkpoint at: {str(RESUME_PATH).replace(prefix, '')}")
    print(f"[SYSTEM] Resuming training...")
    checkpoint = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    if 'relobralo_state' in checkpoint:
        relobralo.load_state_dict(checkpoint['relobralo_state'])
        print(f"[SYSTEM] ReLoBRaLo restored: λ_data={relobralo.weights[0]:.6f}, λ_phys={relobralo.weights[1]:.6f}")
    torch.set_rng_state(checkpoint['torch_rng'].cpu().to(torch.uint8) if isinstance(checkpoint['torch_rng'], torch.Tensor) else torch.ByteTensor(checkpoint['torch_rng']))
    np.random.set_state(checkpoint['numpy_rng'])
    if torch.cuda.is_available() and 'cuda_rng' in checkpoint and checkpoint['cuda_rng'] is not None:
        cuda_rng = checkpoint['cuda_rng']
        if isinstance(cuda_rng, torch.Tensor):
            cuda_rng = cuda_rng.cpu().to(torch.uint8)
        else:
            cuda_rng = torch.ByteTensor(cuda_rng)
        torch.cuda.set_rng_state(cuda_rng)
    print(f"[SYSTEM] Resumed from Epoch {checkpoint['epoch']}. Best Val Loss: {best_val_loss:.6f}")
else:
    print(f"\n[SYSTEM] No checkpoint found. Starting fresh training.")

gpu_log_file = open(gpu_log_path, "w")
gpu_logger = subprocess.Popen(
    ["nvidia-smi", "-i", str(GPU_ID), "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
     "--format=csv", "-l", "5"],
    stdout=gpu_log_file, stderr=subprocess.STDOUT
)

csv_mode = 'a' if RESUME_PATH.exists() and epoch_log_path.exists() else 'w'
epoch_log_file = open(epoch_log_path, mode=csv_mode, newline='')
epoch_writer = csv.writer(epoch_log_file)
if csv_mode == 'w':
    epoch_writer.writerow([
        'Epoch', 'Train_Loss_Total', 'Train_Loss_Data', 'Train_Loss_Phys',
        'Val_Loss_Total', 'Val_Loss_Data', 'Val_Loss_Phys',
        'Lambda_Data', 'Lambda_Phys',
        'C0', 'C1', 'h0', 'h1', 'k01', 'k10', 'q0', 'q1', 'T_amb'
    ])

# ==========================================
# 7. PRE-TRAINING DATA INSPECTION
# ==========================================
print(f"\n[INSPECT] ========== FIRST BATCH DEEP INSPECTION ==========")
_inspect_keys, _inspect_block = next(iter(train_loader))
print(f"[INSPECT] raw data_block  : shape={list(_inspect_block.shape)}, dtype={_inspect_block.dtype}")
print(f"[INSPECT]   Interpretation: ({_inspect_block.shape[0]} samples) x ({_inspect_block.shape[1]} timesteps) x ({_inspect_block.shape[2]} features)")
print(f"[INSPECT]   Features: {scalars['feature_order']}")
print(f"[INSPECT] First sample, first 3 timesteps (RAW ABSOLUTE):")
for _t in range(min(3, _inspect_block.shape[1])):
    _row = _inspect_block[0, _t, :].numpy()
    print(f"[INSPECT]   t={_t}:")
    print(f"[INSPECT]       Power : GPU0={_row[0]:>7.4f}W | GPU1={_row[1]:>7.4f}W")
    print(f"[INSPECT]       Util  : GPU0={_row[2]:>7.4f}% | GPU1={_row[3]:>7.4f}%")
    print(f"[INSPECT]       Temp  : GPU0={_row[4]:>7.4f}C | GPU1={_row[5]:>7.4f}C")

_X_seq = _inspect_block[:, :-1, :]
_Y_true_abs = _inspect_block[:, -1, 4:6]
_Y_true_norm = (_Y_true_abs - mins[4:6].cpu()) / ranges[4:6].cpu()
_X_norm = (_X_seq - mins.cpu()) / ranges.cpu()

print(f"\n[INSPECT] After slicing & normalization:")
print(f"[INSPECT]   X_seq (input)      : shape={list(_X_seq.shape)}, dtype={_X_seq.dtype}")
print(f"[INSPECT]   Y_true_abs (target): shape={list(_Y_true_abs.shape)}, dtype={_Y_true_abs.dtype}")
print(f"[INSPECT]   X_norm (normalized): shape={list(_X_norm.shape)}, dtype={_X_norm.dtype}")
print(f"[INSPECT]   Y_true_norm        : shape={list(_Y_true_norm.shape)}, dtype={_Y_true_norm.dtype}")
print(f"[INSPECT] Sample Row 0:")
print(f"[INSPECT]   Absolute   : T0={_Y_true_abs[0, 0].item():.4f}, T1={_Y_true_abs[0, 1].item():.4f}")
print(f"[INSPECT]   Normalized : T0={_Y_true_norm[0, 0].item():.6f}, T1={_Y_true_norm[0, 1].item():.6f}")
print(f"[INSPECT] ========== END FIRST BATCH INSPECTION ==========\n")

del _inspect_keys, _inspect_block, _X_seq, _Y_true_abs, _Y_true_norm, _X_norm, _row

print(f"\n{'='*80}")
print(f"[TRAINING] INITIATING HPC MODEL TRAINING [{VERSION_PREFIX}]")
print(f"[TRAINING] Using ReLoBRaLo adaptive loss balancing")
print(f"[TRAINING] Physics loss: INTEGRATED FORM in normalized [0,1] space")
print(f"[TRAINING] Epochs: {start_epoch} → {EPOCHS} | Device: {device}")
print(f"{'='*80}")

# ==========================================
# 8. MAIN EPOCH LOOP
# ==========================================
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start_time = time.time()
        model.train()

        total_train_loss = 0.0
        total_train_loss_data = 0.0
        total_train_loss_phys = 0.0
        num_train_batches = 0

        train_pbar = tqdm(train_loader, total=TOTAL_TRAIN_BATCHES, desc=f"Epoch {epoch:02d}/{EPOCHS} [Train]", leave=False)

        for batch_idx, (keys, data_block) in enumerate(train_pbar):
            X_seq = data_block[:, :-1, :].to(device)
            Y_true_abs = data_block[:, -1, 4:6].to(device)
            Y_true_norm = (Y_true_abs - mins[4:6]) / ranges[4:6]
            X_norm = (X_seq - mins) / ranges

            optimizer.zero_grad()

            # --- Forward pass (FP16 for data loss) ---
            with torch.amp.autocast('cuda'):
                Y_pred_norm = model(X_norm)
                loss_data = mse_loss(Y_pred_norm, Y_true_norm)

            # --- Physics loss: integrated form in normalized space (FP32) ---
            T_phys_norm, _ = compute_physics_targets_norm(model, X_seq, DT, mins, ranges)
            loss_phys = mse_loss(Y_pred_norm.float(), T_phys_norm)

            # --- ReLoBRaLo: get adaptive weights ---
            lambdas = relobralo.get_weights([loss_data.item(), loss_phys.item()])
            lambda_data = lambdas[0]
            lambda_phys = lambdas[1]

            # --- Composite loss ---
            loss_total = lambda_data * loss_data + lambda_phys * loss_phys

            scaler.scale(loss_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss_total.item()
            total_train_loss_data += loss_data.item()
            total_train_loss_phys += loss_phys.item()
            num_train_batches += 1

            train_pbar.set_postfix({
                "Loss": f"{loss_total.item():.4f}",
                "Ld": f"{loss_data.item():.2e}",
                "Lp": f"{loss_phys.item():.2e}",
                "λd": f"{lambda_data:.3f}",
                "λp": f"{lambda_phys:.3f}"
            })

        avg_train_loss = total_train_loss / num_train_batches
        avg_train_data = total_train_loss_data / num_train_batches
        avg_train_phys = total_train_loss_phys / num_train_batches

        # --- VALIDATION LOOP ---
        model.eval()
        total_val_loss = 0.0
        total_val_loss_data = 0.0
        total_val_loss_phys = 0.0
        num_val_batches = 0

        val_pbar = tqdm(val_loader, total=TOTAL_VAL_BATCHES, desc="Validation", leave=False)

        with torch.no_grad():
            for batch_idx, (keys, val_block) in enumerate(val_pbar):
                X_val = val_block[:, :-1, :].to(device)
                Y_val_true_abs = val_block[:, -1, 4:6].to(device)
                Y_val_true_norm = (Y_val_true_abs - mins[4:6]) / ranges[4:6]
                X_val_norm = (X_val - mins) / ranges

                with torch.amp.autocast('cuda'):
                    Y_val_pred_norm = model(X_val_norm)
                    loss_data_v = mse_loss(Y_val_pred_norm, Y_val_true_norm)

                T_phys_norm_v, _ = compute_physics_targets_norm(model, X_val, DT, mins, ranges)
                loss_phys_v = mse_loss(Y_val_pred_norm.float(), T_phys_norm_v)

                loss_data_v_scalar = loss_data_v.item()
                loss_phys_v_scalar = loss_phys_v.item()
                loss_total_v = lambda_data * loss_data_v_scalar + lambda_phys * loss_phys_v_scalar

                total_val_loss += loss_total_v
                total_val_loss_data += loss_data_v_scalar
                total_val_loss_phys += loss_phys_v_scalar
                num_val_batches += 1

        avg_val_loss = total_val_loss / num_val_batches
        avg_val_data = total_val_loss_data / num_val_batches
        avg_val_phys = total_val_loss_phys / num_val_batches
        epoch_duration = time.time() - epoch_start_time

        current_phys = model.get_bounded_physics()
        epoch_writer.writerow([
            epoch, avg_train_loss, avg_train_data, avg_train_phys,
            avg_val_loss, avg_val_data, avg_val_phys,
            lambda_data, lambda_phys,
            *[current_phys[k].item() for k in ['C0','C1','h0','h1','k01','k10','q0','q1','T_amb']]
        ])
        epoch_log_file.flush()

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_flag = f" [LR Reduced to {new_lr:.2e}]" if new_lr < old_lr else ""
        best_flag = ""

        checkpoint_state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_val_loss': best_val_loss,
            'relobralo_state': relobralo.state_dict(),
            'torch_rng': torch.get_rng_state(),
            'numpy_rng': np.random.get_state(),
            'cuda_rng': torch.cuda.get_rng_state() if torch.cuda.is_available() else None
        }
        torch.save(checkpoint_state, RESUME_PATH)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            checkpoint_state['best_val_loss'] = best_val_loss
            torch.save(checkpoint_state, BEST_PATH)
            best_flag = " [*]"

        if epoch == 1:
            epoch1_walkthrough(
                model=model, relobralo=relobralo, current_phys=current_phys, DT=DT,
                num_train_batches=num_train_batches,
                total_train_loss=total_train_loss, total_train_loss_data=total_train_loss_data,
                total_train_loss_phys=total_train_loss_phys,
                avg_train_loss=avg_train_loss, avg_train_data=avg_train_data, avg_train_phys=avg_train_phys,
                num_val_batches=num_val_batches,
                total_val_loss=total_val_loss, total_val_loss_data=total_val_loss_data,
                total_val_loss_phys=total_val_loss_phys,
                avg_val_loss=avg_val_loss, avg_val_data=avg_val_data, avg_val_phys=avg_val_phys,
                lambda_data=lambda_data, lambda_phys=lambda_phys,
                Y_val_pred_norm=Y_val_pred_norm, Y_val_true_norm=Y_val_true_norm,
                T_phys_norm_v=T_phys_norm_v,
                X_val=X_val, loss_data_v=loss_data_v, loss_phys_v=loss_phys_v,
                mins_cpu=mins.cpu(), ranges_cpu=ranges.cpu()
            )

        print(f"[TRAINING] Epoch {epoch:02d}/{EPOCHS} | Time: {epoch_duration:.1f}s | "
              f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | "
              f"λd:{lambda_data:.3f} λp:{lambda_phys:.3f}{best_flag}{lr_flag}")

except KeyboardInterrupt:
    print("\n[!] Training manually interrupted. Checkpoints are safe.")
finally:
    try:
        gpu_logger.kill()
    except ProcessLookupError:
        pass
    gpu_log_file.close()
    epoch_log_file.close()
    print(f"[SYSTEM] Background GPU logging terminated. Epoch logs securely saved.")



[SYSTEM] Live GPU Status:
Fri Mar 27 16:24:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 30%   48C    P8             18W /  200W |     191MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.
[SYSTEM] PyTorch initialized. Active Device: NVIDIA RTX A4500 (cuda:1)
[SYSTEM] CUDA Version: 12.6 | PyTorch Version: 2.9.1+cu126
[SYSTEM] GPU Memory: 19.6 GB
[SYSTEM] All RNG sources seeded with MASTER_SEED=42
[SYSTEM] cudnn.deterministic=True, cudnn.benchmark=False



Enter version prefix (e.g., tcn_v1) [Press Enter for default 'unknown_version']:  TCN_ReLoBRaLo_v2



[PATHS] PROJECT_DIR     : Capstone/Implementation
[PATHS] LOCAL_SHARDS_DIR: Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards
[PATHS] SAVE_DIR        : Capstone/Implementation/models/TCN_ReLoBRaLo_v2
[PATHS] SCALARS_PATH    : Capstone/Implementation/data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards/normalization_scalars.json

[DATA] Normalization scalars loaded successfully.
[DATA] Feature Order: ['power_draw_gpu_0_W', 'power_draw_gpu_1_W', 'utilization_gpu_0_pct', 'utilization_gpu_1_pct', 'temperature_gpu_0', 'temperature_gpu_1']
[DATA] Number of Features: 6
[DATA]   [0]             power_draw_gpu_0_W | min=     21.8100 | max=    331.0800
[DATA]   [1]             power_draw_gpu_1_W | min=     22.7800 | max=    328.8600
[DATA]   [2]          utilization_gpu_0_pct | min=      0.0000 | max=    100.0000
[DATA]   [3]          utilization_gpu_1_pct | min=      0.0000 | max=    100.0000


[!] Training manually interrupted. Checkpoints are safe.
[SYSTEM] Background GPU logging terminated. Epoch logs securely saved.


# Testing

In [4]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import numpy as np
import webdataset as wds
import json
import time
from pathlib import Path
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ==========================================
# 1. HARDWARE INITIALIZATION
# ==========================================
def select_gpu():
    """Selects GPU via CLI arg, env var, or interactive prompt."""
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID env var.")
            return gpu_id
    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found.")
    while True:
        gpu_id = input("\n[SYSTEM] Enter GPU ID [Press Enter for '0']: ").strip()
        if not gpu_id:
            return 0
        if gpu_id.isdigit():
            gid = int(gpu_id)
            if gid < torch.cuda.device_count():
                return gid
            print(f"[!] GPU {gid} not found. Available: 0-{torch.cuda.device_count()-1}")
        else:
            print("[!] Invalid input.")

GPU_ID = select_gpu()
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")
if torch.cuda.is_available():
    print(f"[SYSTEM] Active Device: {torch.cuda.get_device_name(GPU_ID)} (cuda:{GPU_ID})")
else:
    print(f"[WARNING] No CUDA device found. Using CPU.")
    device = torch.device("cpu")

# ==========================================
# 2. PATHS & VERSION SELECTION
# ==========================================
PROJECT_DIR = Path(os.path.expanduser("~/Capstone/Implementation"))
LOCAL_SHARDS_DIR = PROJECT_DIR / "data/mit-supercloud-dataset/gpu/dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_shards"
SCALARS_PATH = LOCAL_SHARDS_DIR / "normalization_scalars.json"
MODELS_DIR = PROJECT_DIR / "models"

prefix = "/home/sanke24839/"

print(f"\n[SYSTEM] Available model versions in {str(MODELS_DIR).replace(prefix, '')}:")
available_versions = []
if MODELS_DIR.exists():
    for d in sorted(MODELS_DIR.iterdir()):
        if d.is_dir():
            best_path = d / f"{d.name}_best_model.pt"
            status = "✓ best model found" if best_path.exists() else "✗ no best model"
            available_versions.append(d.name)
            print(f"  [{len(available_versions)}] {d.name} ({status})")
if not available_versions:
    print("  (none found)")

version_input = input("\nEnter version prefix to test (e.g., tcn_v1): ").strip()
if not version_input:
    print("[ERROR] No version prefix provided. Exiting.")
    sys.exit(1)

VERSION_PREFIX = version_input
SAVE_DIR = MODELS_DIR / VERSION_PREFIX
BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
RESULTS_DIR = SAVE_DIR / "test_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not BEST_PATH.exists():
    print(f"[ERROR] Best model checkpoint not found at: {str(BEST_PATH).replace(prefix, '')}")
    sys.exit(1)

print(f"\n[SYSTEM] Loading version: {VERSION_PREFIX}")
print(f"[SYSTEM] Checkpoint: {str(BEST_PATH).replace(prefix, '')}")
print(f"[SYSTEM] Results will be saved to: {str(RESULTS_DIR).replace(prefix, '')}")

# ==========================================
# 3. MODEL ARCHITECTURE (must match training)
# ==========================================
DEFAULT_EMPIRICAL = {
    "C0": 143.22, "C1": 140.25, "h0": 4.8713, "h1": 5.3871,
    "k01": 0.078038, "k10": 0.028120, "q0": 35.0, "q1": 35.0,
    "T_amb": 30.5
}

DEFAULT_FEATURE_INDICES = {"P0": 0, "P1": 1, "T0": 4, "T1": 5}
DT = 0.11


class DilatedTCN(nn.Module):
    def __init__(self, num_inputs=6, num_channels=[32, 64, 128], kernel_size=3):
        super(DilatedTCN, self).__init__()
        layers = []
        in_channels = num_inputs
        for i, out_channels in enumerate(num_channels):
            dilation_size = 2 ** i
            padding = (kernel_size - 1) * dilation_size
            layers += [
                nn.ConstantPad1d((padding, 0), 0.0),
                nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
            ]
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1])


class PINNEngine(nn.Module):
    def __init__(self, empirical_params=None, feature_indices=None):
        super(PINNEngine, self).__init__()
        self.tcn = DilatedTCN()
        self.empirical = empirical_params if empirical_params is not None else dict(DEFAULT_EMPIRICAL)
        self.feature_indices = feature_indices if feature_indices is not None else dict(DEFAULT_FEATURE_INDICES)
        
        self.IDX_P0 = self.feature_indices["P0"]
        self.IDX_P1 = self.feature_indices["P1"]
        self.IDX_T0 = self.feature_indices["T0"]
        self.IDX_T1 = self.feature_indices["T1"]
        
        self.raw_params = nn.ParameterDict()
        for k in self.empirical:
            if k in ('q0', 'q1'):
                init_val = float(np.log(np.exp(self.empirical[k]) - 1.0))
            else:
                init_val = 0.0
            self.raw_params[k] = nn.Parameter(torch.tensor(init_val))

    def get_bounded_physics(self):
        phys = {}
        for k, exact_val in self.empirical.items():
            if k in ('q0', 'q1'):
                phys[k] = nn.functional.softplus(self.raw_params[k])
            elif k == 'T_amb':
                phys[k] = 28.0 + (33.0 - 28.0) * torch.sigmoid(self.raw_params[k])
            else:
                min_bound = exact_val * 0.90
                max_bound = exact_val * 1.10
                phys[k] = min_bound + (max_bound - min_bound) * torch.sigmoid(self.raw_params[k])
        return phys

    def forward(self, x):
        return self.tcn(x)


# ==========================================
# 4. LOAD MODEL & DATA
# ==========================================
model = PINNEngine(empirical_params=DEFAULT_EMPIRICAL, feature_indices=DEFAULT_FEATURE_INDICES).to(device)
checkpoint = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

trained_epoch = checkpoint['epoch']
best_val_loss = checkpoint['best_val_loss']
print(f"\n[MODEL] Loaded best model from Epoch {trained_epoch}")
print(f"[MODEL] Best validation loss: {best_val_loss:.6f}")
print(f"[MODEL] Total parameters: {sum(p.numel() for p in model.parameters()):,}")

learned_phys = model.get_bounded_physics()
print(f"\n[MODEL] Learned Physics Parameters:")
print(f"  {'Param':>5s} | {'Learned':>12s} | {'Empirical':>12s} | {'Δ%':>8s} | {'Type'}")
print(f"  {'-'*5} | {'-'*12} | {'-'*12} | {'-'*8} | {'-'*15}")
for k in DEFAULT_EMPIRICAL:
    learned_val = learned_phys[k].item()
    emp_val = DEFAULT_EMPIRICAL[k]
    pct = ((learned_val / emp_val) - 1) * 100
    if k in ('q0', 'q1'):
        ptype = "softplus (unbounded)"
    elif k == 'T_amb':
        ptype = "sigmoid ([28.0, 33.0])"
    else:
        ptype = "sigmoid (±10%)"
    print(f"  {k:>5s} | {learned_val:>12.6f} | {emp_val:>12.6f} | {pct:>+7.2f}% | {ptype}")

with open(SCALARS_PATH, 'r') as f:
    scalars = json.load(f)
mins = torch.tensor([scalars['global_minimums'][c] for c in scalars['feature_order']], device=device)
maxs = torch.tensor([scalars['global_maximums'][c] for c in scalars['feature_order']], device=device)
ranges = maxs - mins
ranges[ranges == 0] = 1.0
print(f"\n[DATA] Normalization scalars loaded. Features: {scalars['feature_order']}")

test_files = [str(p) for p in sorted((LOCAL_SHARDS_DIR / "test").glob("*.tar"))]
if not test_files:
    print(f"[ERROR] No test shards found in {LOCAL_SHARDS_DIR / 'test'}")
    sys.exit(1)
print(f"[DATA] Test shards: {len(test_files)} files")

BATCH_SIZE = 16384
NUM_WORKERS = 8
TOTAL_VAL_BATCHES = 1359

def make_windows_vectorized(src, window_size=50, stride=10):
    for key, npy_array in src:
        n_rows = npy_array.shape[0]
        if n_rows < window_size:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(npy_array, window_shape=window_size, axis=0)
        windows = np.swapaxes(windows, 1, 2)
        windows = windows[::stride].copy()
        for w in windows:
            yield key, w


def create_test_dataloader(shard_list):
    dataset = wds.WebDataset(shard_list, shardshuffle=0).decode().to_tuple("__key__", "data.npy")
    dataset = dataset.compose(lambda src: make_windows_vectorized(src, window_size=50, stride=10))
    dataset = dataset.batched(BATCH_SIZE)
    return torch.utils.data.DataLoader(dataset, batch_size=None, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)


test_loader = create_test_dataloader(test_files)


# ==========================================
# 5. INTEGRATED-FORM PHYSICS HELPER
# ==========================================
def compute_physics_targets_abs(model, X_seq_abs, DT):
    """
    Computes physics-predicted next temperature in ABSOLUTE °C.
    T_next = T_current + DT * (1/C) * (P + k*P' - h*(T - T_amb) + q)
    """
    phys = model.get_bounded_physics()
    X_last = X_seq_abs[:, -1, :].float()
    P0 = X_last[:, model.IDX_P0]
    P1 = X_last[:, model.IDX_P1]
    T0_t = X_last[:, model.IDX_T0]
    T1_t = X_last[:, model.IDX_T1]

    T0_phys = T0_t + DT * (1 / phys["C0"]) * (
        P0 + phys["k01"] * P1 - phys["h0"] * (T0_t - phys["T_amb"]) + phys["q0"]
    )
    T1_phys = T1_t + DT * (1 / phys["C1"]) * (
        P1 + phys["k10"] * P0 - phys["h1"] * (T1_t - phys["T_amb"]) + phys["q1"]
    )

    T_phys_abs = torch.stack([T0_phys, T1_phys], dim=1)  # [B, 2]
    return T_phys_abs, phys


# ==========================================
# 6. INFERENCE LOOP (GPU-accelerated)
# ==========================================
print(f"\n{'='*70}")
print(f"[TEST] Starting inference on test set...")
print(f"{'='*70}")

mse_loss = nn.MSELoss()

all_y_true_abs = []
all_y_pred_abs = []
all_t_phys_abs = []   # physics-predicted temperatures
all_keys = []

total_test_loss_data = 0.0
total_test_loss_phys = 0.0
num_test_batches = 0
inference_start = time.time()

with torch.no_grad():
    for batch_idx, (keys, data_block) in enumerate(tqdm(test_loader, total=TOTAL_VAL_BATCHES, desc="Testing")):
        X_seq = data_block[:, :-1, :].to(device)
        Y_true_abs = data_block[:, -1, 4:6].to(device)
        Y_true_norm = (Y_true_abs - mins[4:6]) / ranges[4:6]
        X_norm = (X_seq - mins) / ranges

        with torch.amp.autocast('cuda'):
            Y_pred_norm = model(X_norm)
            loss_data = mse_loss(Y_pred_norm, Y_true_norm)

        # Data predictions in absolute
        Y_pred_abs_fp32 = Y_pred_norm.float() * ranges[4:6] + mins[4:6]

        # Physics targets (absolute) and physics loss (normalized space)
        T_phys_abs, _ = compute_physics_targets_abs(model, X_seq, DT)
        T_phys_norm = (T_phys_abs - mins[4:6]) / ranges[4:6]
        loss_phys = mse_loss(Y_pred_norm.float(), T_phys_norm)

        all_y_true_abs.append(Y_true_abs.cpu())
        all_y_pred_abs.append(Y_pred_abs_fp32.cpu())
        all_t_phys_abs.append(T_phys_abs.cpu())
        if isinstance(keys, (list, tuple)):
            all_keys.extend(keys)
        else:
            all_keys.extend(list(keys))

        total_test_loss_data += loss_data.item()
        total_test_loss_phys += loss_phys.item()
        num_test_batches += 1

inference_time = time.time() - inference_start

if num_test_batches == 0:
    print("[ERROR] No test batches were processed.")
    sys.exit(1)

Y_true = torch.cat(all_y_true_abs, dim=0).numpy()
Y_pred = torch.cat(all_y_pred_abs, dim=0).numpy()
T_phys = torch.cat(all_t_phys_abs, dim=0).numpy()

N = Y_true.shape[0]
print(f"\n[TEST] Inference complete: {N:,} samples in {inference_time:.1f}s ({N/inference_time:,.0f} samples/sec)")

del all_y_true_abs, all_y_pred_abs, all_t_phys_abs
torch.cuda.empty_cache()


# ==========================================
# 7. COMPUTE ALL METRICS
# ==========================================
def compute_regression_metrics(y_true, y_pred, name=""):
    errors = y_pred - y_true
    abs_errors = np.abs(errors)
    sq_errors = errors ** 2
    mse = np.mean(sq_errors)
    rmse = np.sqrt(mse)
    mae = np.mean(abs_errors)
    max_ae = np.max(abs_errors)
    ss_res = np.sum(sq_errors)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else float('nan')
    nonzero_mask = np.abs(y_true) > 1e-8
    mape = np.mean(np.abs(errors[nonzero_mask] / y_true[nonzero_mask])) * 100 if nonzero_mask.sum() > 0 else float('nan')
    p50 = np.percentile(abs_errors, 50)
    p90 = np.percentile(abs_errors, 90)
    p95 = np.percentile(abs_errors, 95)
    p99 = np.percentile(abs_errors, 99)
    mean_bias = np.mean(errors)
    std_error = np.std(errors)
    return {
        'name': name, 'mse': mse, 'rmse': rmse, 'mae': mae, 'max_ae': max_ae,
        'r2': r2, 'mape': mape, 'mean_bias': mean_bias, 'std_error': std_error,
        'p50': p50, 'p90': p90, 'p95': p95, 'p99': p99,
        'errors': errors, 'abs_errors': abs_errors,
    }


metrics_T0 = compute_regression_metrics(Y_true[:, 0], Y_pred[:, 0], "T0 (GPU0 Temp)")
metrics_T1 = compute_regression_metrics(Y_true[:, 1], Y_pred[:, 1], "T1 (GPU1 Temp)")
metrics_overall = compute_regression_metrics(Y_true.flatten(), Y_pred.flatten(), "Overall (T0+T1)")

# Physics target agreement: how close model predictions are to physics predictions
phys_agreement_T0 = compute_regression_metrics(T_phys[:, 0], Y_pred[:, 0], "T0 vs Physics Target")
phys_agreement_T1 = compute_regression_metrics(T_phys[:, 1], Y_pred[:, 1], "T1 vs Physics Target")

avg_data_loss = total_test_loss_data / num_test_batches
avg_phys_loss = total_test_loss_phys / num_test_batches

# ==========================================
# 8. PRINT METRICS REPORT
# ==========================================
SEP = "=" * 70
print(f"\n{SEP}")
print(f"  TEST RESULTS REPORT — Version: {VERSION_PREFIX}")
print(f"  Model from Epoch {trained_epoch} | {N:,} test samples")
print(f"  Physics loss: INTEGRATED FORM in normalized [0,1] space")
print(f"{SEP}")

def print_metric_block(m):
    print(f"\n  --- {m['name']} ---")
    print(f"  {'MSE':>20s} : {m['mse']:.8f}")
    print(f"  {'RMSE':>20s} : {m['rmse']:.6f} °C")
    print(f"  {'MAE':>20s} : {m['mae']:.6f} °C")
    print(f"  {'Max Abs Error':>20s} : {m['max_ae']:.6f} °C")
    print(f"  {'R²':>20s} : {m['r2']:.8f}")
    print(f"  {'MAPE':>20s} : {m['mape']:.4f} %")
    print(f"  {'Mean Bias':>20s} : {m['mean_bias']:+.6f} °C")
    print(f"  {'Std of Error':>20s} : {m['std_error']:.6f} °C")
    print(f"  {'P50 Abs Error':>20s} : {m['p50']:.6f} °C")
    print(f"  {'P90 Abs Error':>20s} : {m['p90']:.6f} °C")
    print(f"  {'P95 Abs Error':>20s} : {m['p95']:.6f} °C")
    print(f"  {'P99 Abs Error':>20s} : {m['p99']:.6f} °C")

print_metric_block(metrics_T0)
print_metric_block(metrics_T1)
print_metric_block(metrics_overall)

print(f"\n  --- Physics Target Agreement (Pred vs Physics ODE Target, in °C) ---")
print(f"  T0: RMSE={phys_agreement_T0['rmse']:.6f}°C, MAE={phys_agreement_T0['mae']:.6f}°C, Bias={phys_agreement_T0['mean_bias']:+.6f}°C")
print(f"  T1: RMSE={phys_agreement_T1['rmse']:.6f}°C, MAE={phys_agreement_T1['mae']:.6f}°C, Bias={phys_agreement_T1['mean_bias']:+.6f}°C")

print(f"\n  --- Loss Values (normalized [0,1] space) ---")
print(f"  {'Avg Data Loss':>20s} : {avg_data_loss:.8f}")
print(f"  {'Avg Physics Loss':>20s} : {avg_phys_loss:.8f}")
print(f"  {'Ratio (phys/data)':>20s} : {avg_phys_loss / max(avg_data_loss, 1e-12):.2f}x")

print(f"\n  --- Learned Physics Parameters ---")
print(f"  {'Param':>5s} | {'Learned':>12s} | {'Empirical':>12s} | {'Δ%':>8s}")
print(f"  {'-'*5} | {'-'*12} | {'-'*12} | {'-'*8}")
for k in DEFAULT_EMPIRICAL:
    lv = learned_phys[k].item()
    ev = DEFAULT_EMPIRICAL[k]
    pct = ((lv / ev) - 1) * 100
    print(f"  {k:>5s} | {lv:>12.6f} | {ev:>12.6f} | {pct:>+7.2f}%")
print(f"{SEP}")


# ==========================================
# 9. VISUALIZATION
# ==========================================
print(f"\n[PLOTS] Generating diagnostic plots...")
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

# --- PLOT 1: Predicted vs Actual Scatter ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, y_t, y_p, label in zip(axes, [Y_true[:, 0], Y_true[:, 1]], [Y_pred[:, 0], Y_pred[:, 1]], ['T0 (GPU0)', 'T1 (GPU1)']):
    n_plot = min(50000, len(y_t))
    idx = np.random.choice(len(y_t), n_plot, replace=False)
    ax.scatter(y_t[idx], y_p[idx], alpha=0.15, s=3, c='steelblue', rasterized=True)
    lims = [min(y_t.min(), y_p.min()) - 1, max(y_t.max(), y_p.max()) + 1]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Temperature (°C)'); ax.set_ylabel('Predicted Temperature (°C)')
    ax.set_title(f'{label} — Predicted vs Actual'); ax.set_aspect('equal'); ax.legend(loc='upper left')
fig.suptitle(f'{VERSION_PREFIX} — Predicted vs Actual', fontsize=14, fontweight='bold')
fig.tight_layout(); fig.savefig(RESULTS_DIR / '01_pred_vs_actual.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 01_pred_vs_actual.png")

# --- PLOT 2: Error Distribution Histograms ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, m, color in zip(axes, [metrics_T0, metrics_T1], ['#2196F3', '#FF9800']):
    ax.hist(m['errors'], bins=200, density=True, alpha=0.7, color=color, edgecolor='none')
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.axvline(m['mean_bias'], color='darkred', linestyle='-', linewidth=1.5, label=f"Mean bias={m['mean_bias']:+.4f}°C")
    ax.set_xlabel('Prediction Error (°C)'); ax.set_ylabel('Density')
    ax.set_title(f"{m['name']} — Error Distribution\nRMSE={m['rmse']:.4f}°C, MAE={m['mae']:.4f}°C"); ax.legend()
fig.suptitle(f'{VERSION_PREFIX} — Error Distributions', fontsize=14, fontweight='bold')
fig.tight_layout(); fig.savefig(RESULTS_DIR / '02_error_distributions.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 02_error_distributions.png")

# --- PLOT 3: Residual vs Predicted ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_p, m, color in zip(axes, [Y_pred[:, 0], Y_pred[:, 1]], [metrics_T0, metrics_T1], ['#2196F3', '#FF9800']):
    n_plot = min(50000, len(y_p))
    idx = np.random.choice(len(y_p), n_plot, replace=False)
    ax.scatter(y_p[idx], m['errors'][idx], alpha=0.15, s=3, c=color, rasterized=True)
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Predicted Temperature (°C)'); ax.set_ylabel('Residual Error (°C)')
    ax.set_title(f"{m['name']} — Residuals vs Predicted")
fig.suptitle(f'{VERSION_PREFIX} — Residual Analysis', fontsize=14, fontweight='bold')
fig.tight_layout(); fig.savefig(RESULTS_DIR / '03_residual_vs_predicted.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 03_residual_vs_predicted.png")

# --- PLOT 4: Absolute Error CDF ---
fig, ax = plt.subplots(figsize=(10, 6))
for m, color in zip([metrics_T0, metrics_T1], ['#2196F3', '#FF9800']):
    sorted_ae = np.sort(m['abs_errors'])
    cdf = np.arange(1, len(sorted_ae) + 1) / len(sorted_ae)
    step = max(1, len(sorted_ae) // 5000)
    ax.plot(sorted_ae[::step], cdf[::step], label=m['name'], color=color, linewidth=1.5)
for pctl, label in [(0.90, 'P90'), (0.95, 'P95'), (0.99, 'P99')]:
    ax.axhline(pctl, color='gray', linestyle=':', linewidth=0.8, alpha=0.7)
    ax.text(ax.get_xlim()[1] * 0.95, pctl + 0.005, label, ha='right', fontsize=9, color='gray')
ax.set_xlabel('Absolute Error (°C)'); ax.set_ylabel('Cumulative Probability')
ax.set_title(f'{VERSION_PREFIX} — Absolute Error CDF'); ax.legend(); ax.set_xlim(left=0)
fig.tight_layout(); fig.savefig(RESULTS_DIR / '04_error_cdf.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 04_error_cdf.png")

# --- PLOT 5: Physics Target vs Actual & Model (3-panel) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Show how close PHYSICS targets are to ACTUAL temperatures
phys_err_T0 = T_phys[:, 0] - Y_true[:, 0]
phys_err_T1 = T_phys[:, 1] - Y_true[:, 1]
for ax, pe, label, color in zip(axes, [phys_err_T0, phys_err_T1], ['T0 (GPU0)', 'T1 (GPU1)'], ['#4CAF50', '#9C27B0']):
    ax.hist(pe, bins=200, density=True, alpha=0.7, color=color, edgecolor='none')
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    mu, sigma = np.mean(pe), np.std(pe)
    ax.axvline(mu, color='darkred', linestyle='-', linewidth=1.5, label=f'Mean={mu:+.4f}°C')
    ax.set_xlabel('Physics Target Error (°C)'); ax.set_ylabel('Density')
    ax.set_title(f'{label} — Physics Target vs Actual\nμ={mu:+.4f}°C, σ={sigma:.4f}°C'); ax.legend()
fig.suptitle(f'{VERSION_PREFIX} — Physics ODE Target Accuracy', fontsize=14, fontweight='bold')
fig.tight_layout(); fig.savefig(RESULTS_DIR / '05_physics_target_accuracy.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 05_physics_target_accuracy.png")

# --- PLOT 6: Learned vs Empirical Physics Parameters ---
fig, ax = plt.subplots(figsize=(12, 5))
param_names = list(DEFAULT_EMPIRICAL.keys())
learned_vals = [learned_phys[k].item() for k in param_names]
empirical_vals = [DEFAULT_EMPIRICAL[k] for k in param_names]
x = np.arange(len(param_names)); width = 0.35
ax.bar(x - width/2, empirical_vals, width, label='Empirical', color='#90CAF9', edgecolor='#1565C0')
ax.bar(x + width/2, learned_vals, width, label='Learned', color='#FF8A65', edgecolor='#BF360C')
ax.set_xlabel('Physics Parameter'); ax.set_ylabel('Value')
ax.set_title(f'{VERSION_PREFIX} — Learned vs Empirical Physics Parameters')
ax.set_xticks(x); ax.set_xticklabels(param_names); ax.legend()
for i, (ev, lv) in enumerate(zip(empirical_vals, learned_vals)):
    pct = ((lv / ev) - 1) * 100
    ax.annotate(f'{pct:+.1f}%', xy=(x[i] + width/2, lv), ha='center', va='bottom', fontsize=8, fontweight='bold')
fig.tight_layout(); fig.savefig(RESULTS_DIR / '06_physics_params.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 06_physics_params.png")

# --- PLOT 7: Per-Job Time-Series (3-5 random jobs, full trajectory) ---
print(f"\n[PLOTS] Generating per-job time-series plots...")

keys_array = np.array(all_keys)
job_ids = np.array([k.rsplit('/', 1)[0] if '/' in k else k.rsplit('.', 1)[0] if '.' in k else k for k in all_keys])
unique_jobs = np.unique(job_ids)
print(f"[PLOTS] Found {len(unique_jobs)} unique jobs in test set")

np.random.seed(42)
candidate_jobs = []
for job in unique_jobs:
    n_samples = np.sum(job_ids == job)
    if n_samples >= 50:
        candidate_jobs.append((job, n_samples))

candidate_jobs.sort(key=lambda x: x[1])
if len(candidate_jobs) > 5:
    indices = np.linspace(0, len(candidate_jobs) - 1, 5, dtype=int)
    selected_jobs = [candidate_jobs[i] for i in indices]
else:
    selected_jobs = candidate_jobs[:5]

print(f"[PLOTS] Selected {len(selected_jobs)} jobs for time-series:")
for job_name, n_samp in selected_jobs:
    print(f"  - {job_name} ({n_samp} samples)")

for job_idx, (job_name, n_samp) in enumerate(selected_jobs):
    mask = job_ids == job_name
    job_true = Y_true[mask]
    job_pred = Y_pred[mask]
    job_phys = T_phys[mask]

    fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

    # GPU0
    axes[0].plot(range(len(job_true)), job_true[:, 0], color='#1565C0', linewidth=1.2,
                 label='Actual T0', alpha=0.9, zorder=2)
    axes[0].plot(range(len(job_pred)), job_pred[:, 0], color='#E53935', linewidth=1.0,
                 label='Predicted T0', linestyle='--', alpha=0.85, zorder=3)
    axes[0].plot(range(len(job_phys)), job_phys[:, 0], color='#43A047', linewidth=0.8,
                 label='Physics Target T0', linestyle=':', alpha=0.6, zorder=1)
    axes[0].fill_between(range(len(job_true)), job_true[:, 0], job_pred[:, 0],
                         alpha=0.12, color='red', label='Error band')
    axes[0].set_ylabel('Temperature (°C)')
    axes[0].set_title(f'GPU 0 — Actual vs Predicted vs Physics Target')
    axes[0].legend(loc='upper right')

    # GPU1
    axes[1].plot(range(len(job_true)), job_true[:, 1], color='#E65100', linewidth=1.2,
                 label='Actual T1', alpha=0.9, zorder=2)
    axes[1].plot(range(len(job_pred)), job_pred[:, 1], color='#7B1FA2', linewidth=1.0,
                 label='Predicted T1', linestyle='--', alpha=0.85, zorder=3)
    axes[1].plot(range(len(job_phys)), job_phys[:, 1], color='#43A047', linewidth=0.8,
                 label='Physics Target T1', linestyle=':', alpha=0.6, zorder=1)
    axes[1].fill_between(range(len(job_true)), job_true[:, 1], job_pred[:, 1],
                         alpha=0.12, color='purple', label='Error band')
    axes[1].set_ylabel('Temperature (°C)')
    axes[1].set_xlabel('Sample Index (within job)')
    axes[1].set_title(f'GPU 1 — Actual vs Predicted vs Physics Target')
    axes[1].legend(loc='upper right')

    safe_name = job_name.replace('/', '_').replace('\\', '_')
    fig.suptitle(f'{VERSION_PREFIX} — Job: {job_name}\n({n_samp} samples)', fontsize=13, fontweight='bold')
    fig.tight_layout()
    fname = f'07_timeseries_job_{job_idx+1}_{safe_name}.png'
    fig.savefig(RESULTS_DIR / fname, bbox_inches='tight')
    plt.close(fig)
    print(f"  [✓] {fname}")

# --- PLOT 8: Joint Error Heatmap ---
fig, ax = plt.subplots(figsize=(8, 7))
h = ax.hist2d(metrics_T0['errors'], metrics_T1['errors'], bins=150, cmap='inferno', cmin=1)
fig.colorbar(h[3], ax=ax, label='Count')
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.7)
ax.axvline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_xlabel('T0 Error (°C)'); ax.set_ylabel('T1 Error (°C)')
ax.set_title(f'{VERSION_PREFIX} — Joint Error Distribution')
corr = np.corrcoef(metrics_T0['errors'], metrics_T1['errors'])[0, 1]
ax.text(0.05, 0.95, f'Pearson r = {corr:.4f}', transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
fig.tight_layout(); fig.savefig(RESULTS_DIR / '08_joint_error_heatmap.png', bbox_inches='tight'); plt.close(fig)
print(f"  [✓] 08_joint_error_heatmap.png")


# ==========================================
# 10. SAVE METRICS TO JSON
# ==========================================
def sanitize_for_json(m):
    return {k: (float(v) if isinstance(v, (np.floating, float)) else v)
            for k, v in m.items() if not isinstance(v, np.ndarray)}

metrics_export = {
    'version_prefix': VERSION_PREFIX,
    'trained_epoch': int(trained_epoch),
    'best_val_loss': float(best_val_loss),
    'num_test_samples': int(N),
    'inference_time_seconds': round(inference_time, 2),
    'DT': DT,
    'physics_loss_formulation': 'integrated form in normalized [0,1] space',
    'metrics_T0': sanitize_for_json(metrics_T0),
    'metrics_T1': sanitize_for_json(metrics_T1),
    'metrics_overall': sanitize_for_json(metrics_overall),
    'physics_target_agreement_T0': sanitize_for_json(phys_agreement_T0),
    'physics_target_agreement_T1': sanitize_for_json(phys_agreement_T1),
    'avg_data_loss_normalized': float(avg_data_loss),
    'avg_phys_loss_normalized': float(avg_phys_loss),
    'phys_data_loss_ratio': float(avg_phys_loss / max(avg_data_loss, 1e-12)),
    'learned_physics_params': {k: float(learned_phys[k].item()) for k in DEFAULT_EMPIRICAL},
    'empirical_physics_params': DEFAULT_EMPIRICAL,
}

metrics_json_path = RESULTS_DIR / 'test_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_export, f, indent=2)

print(f"\n[SYSTEM] All metrics saved to: {str(metrics_json_path).replace(prefix, '')}")
print(f"[SYSTEM] All plots saved to: {str(RESULTS_DIR).replace(prefix, '')}")
print(f"\n{'='*70}")
print(f"  TESTING COMPLETE — {VERSION_PREFIX}")
print(f"{'='*70}")



[SYSTEM] Live GPU Status:
Thu Mar 26 19:55:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 30%   41C    P8              7W /  200W |    7395MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter GPU ID [Press Enter for '0']:  1


[SYSTEM] Active Device: NVIDIA RTX A4500 (cuda:1)

[SYSTEM] Available model versions in Capstone/Implementation/models:
  [1] .ipynb_checkpoints (✗ no best model)
  [2] TCN_ReLoBRaLo_v1 (✓ best model found)
  [3] TCN_ReLoBRaLo_v2 (✓ best model found)
  [4] lambda_physics_calculation (✗ no best model)
  [5] tcn_v1 (✓ best model found)
  [6] tcn_v2 (✓ best model found)



Enter version prefix to test (e.g., tcn_v1):  TCN_ReLoBRaLo_v2



[SYSTEM] Loading version: TCN_ReLoBRaLo_v2
[SYSTEM] Checkpoint: Capstone/Implementation/models/TCN_ReLoBRaLo_v2/TCN_ReLoBRaLo_v2_best_model.pt
[SYSTEM] Results will be saved to: Capstone/Implementation/models/TCN_ReLoBRaLo_v2/test_results

[MODEL] Loaded best model from Epoch 4
[MODEL] Best validation loss: 0.003430
[MODEL] Total parameters: 32,235

[MODEL] Learned Physics Parameters:
  Param |      Learned |    Empirical |       Δ% | Type
  ----- | ------------ | ------------ | -------- | ---------------
     C0 |   129.153580 |   143.220000 |   -9.82% | sigmoid (±10%)
     C1 |   126.346870 |   140.250000 |   -9.91% | sigmoid (±10%)
     h0 |     5.353109 |     4.871300 |   +9.89% | sigmoid (±10%)
     h1 |     5.920339 |     5.387100 |   +9.90% | sigmoid (±10%)
    k01 |     0.072125 |     0.078038 |   -7.58% | sigmoid (±10%)
    k10 |     0.026257 |     0.028120 |   -6.63% | sigmoid (±10%)
     q0 |    34.695797 |    35.000000 |   -0.87% | softplus (unbounded)
     q1 |    34.6417

Testing: 100%|██████████| 1359/1359 [00:52<00:00, 25.96it/s]



[TEST] Inference complete: 22,187,769 samples in 52.3s (423,859 samples/sec)

  TEST RESULTS REPORT — Version: TCN_ReLoBRaLo_v2
  Model from Epoch 4 | 22,187,769 test samples
  Physics loss: INTEGRATED FORM in normalized [0,1] space

  --- T0 (GPU0 Temp) ---
                   MSE : 5.86425400
                  RMSE : 2.421622 °C
                   MAE : 1.999191 °C
         Max Abs Error : 13.920898 °C
                    R² : 0.96844500
                  MAPE : 4.5602 %
             Mean Bias : -0.321419 °C
          Std of Error : 2.400198 °C
         P50 Abs Error : 1.799805 °C
         P90 Abs Error : 3.783203 °C
         P95 Abs Error : 4.285156 °C
         P99 Abs Error : 5.628906 °C

  --- T1 (GPU1 Temp) ---
                   MSE : 6.34211159
                  RMSE : 2.518355 °C
                   MAE : 1.984149 °C
         Max Abs Error : 19.193848 °C
                    R² : 0.95981377
                  MAPE : 4.3173 %
             Mean Bias : -1.020338 °C
          Std of 

# 